In [27]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_x.csv")

TARGET = "bilissel_performans_skoru"

X = train.drop(columns=["id", TARGET])
y = train[TARGET]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

print("Train:", X.shape)
print("Test:", X_test.shape)

Train: (56000, 22)
Test: (24000, 22)


In [28]:
def prepare_features(X, X_test):
    X_fe = X.copy()
    X_test_fe = X_test.copy()

    log_cols = [
        "uyku_oncesi_kafein_mg",
        "uyku_oncesi_ekran_suresi_dk"
    ]

    for col in log_cols:
        if col in X_fe.columns:
            X_fe[col] = np.log1p(X_fe[col])
            X_test_fe[col] = np.log1p(X_test_fe[col])

    def add_features(df):
        df = df.copy()

        if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

        if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
            df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

        if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
            df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

        if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
            df["dijital_kafein_yuku"] = df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]

        if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
            df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

        return df

    X_fe = add_features(X_fe)
    X_test_fe = add_features(X_test_fe)

    return X_fe, X_test_fe


X_fe, X_test_fe = prepare_features(X, X_test)

print("X_fe:", X_fe.shape)
print("X_test_fe:", X_test_fe.shape)

X_fe: (56000, 27)
X_test_fe: (24000, 27)


In [29]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
from catboost import CatBoostRegressor

def get_catboost_oof(X_fe, y, X_test_fe, cv):
    X_cb = X_fe.copy()
    X_test_cb = X_test_fe.copy()

    cat_cols = X_cb.select_dtypes(include="object").columns.tolist()

    for col in cat_cols:
        X_cb[col] = X_cb[col].fillna("Bilinmiyor").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("Bilinmiyor").astype(str)

    cat_features_idx = [X_cb.columns.get_loc(col) for col in cat_cols]

    oof_preds = np.zeros(len(X_cb))
    test_preds = np.zeros(len(X_test_cb))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
        X_train_fold = X_cb.iloc[train_idx]
        X_val_fold = X_cb.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=1500,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5,
            random_seed=42,
            verbose=200,
            early_stopping_rounds=100
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_idx,
            eval_set=(X_val_fold, y_val_fold),
            use_best_model=True
        )

        val_pred = model.predict(X_val_fold)
        test_pred = model.predict(X_test_cb)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"CatBoost Fold {fold} RMSE:", fold_rmse)

    print("CatBoost Fold RMSE:", fold_scores)
    print("CatBoost Mean RMSE:", np.mean(fold_scores))
    print("CatBoost OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_cat, test_cat, scores_cat = get_catboost_oof(X_fe, y, X_test_fe, cv)

In [ ]:
def get_hgb_oof(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.04,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.1,
            random_state=42
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"HGB Fold {fold} RMSE:", fold_rmse)

    print("HGB Fold RMSE:", fold_scores)
    print("HGB Mean RMSE:", np.mean(fold_scores))
    print("HGB OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_hgb, test_hgb, scores_hgb = get_hgb_oof(X_fe, y, X_test_fe, cv)

HGB Fold 1 RMSE: 1.228499560987444
HGB Fold 2 RMSE: 1.2290440992060874
HGB Fold 3 RMSE: 1.2159707118361172
HGB Fold 4 RMSE: 1.2243227403073043
HGB Fold 5 RMSE: 1.2450227212554872
HGB Fold RMSE: [np.float64(1.228499560987444), np.float64(1.2290440992060874), np.float64(1.2159707118361172), np.float64(1.2243227403073043), np.float64(1.2450227212554872)]
HGB Mean RMSE: 1.228571966718488
HGB OOF RMSE: 1.2286084071060954


In [ ]:
from scipy.optimize import minimize

oof_matrix = np.column_stack([
    oof_cat,
    oof_hgb
])

test_matrix = np.column_stack([
    test_cat,
    test_hgb
])

def blend_rmse(weights):
    preds = oof_matrix @ weights
    return rmse(y, preds)

n_models = oof_matrix.shape[1]

initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result = minimize(
    blend_rmse,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights = result.x
best_oof_preds = oof_matrix @ best_weights
best_test_preds = test_matrix @ best_weights

print("Best weights:", best_weights)
print("Blend OOF RMSE:", rmse(y, best_oof_preds))

Best weights: [0.84412931 0.15587069]
Blend OOF RMSE: 1.2164008947511131


## Deney 12 - LightGBM OOF

In [ ]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [ ]:
def get_lgbm_oof(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = LGBMRegressor(
            objective="regression",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=25,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.05,
            reg_lambda=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"LightGBM Fold {fold} RMSE:", fold_rmse)

    print("LightGBM Fold RMSE:", fold_scores)
    print("LightGBM Mean RMSE:", np.mean(fold_scores))
    print("LightGBM OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_lgbm, test_lgbm, scores_lgbm = get_lgbm_oof(X_fe, y, X_test_fe, cv)

LightGBM Fold 1 RMSE: 1.2309181643011173
LightGBM Fold 2 RMSE: 1.2325901907249033
LightGBM Fold 3 RMSE: 1.216643647877837
LightGBM Fold 4 RMSE: 1.227986278236269
LightGBM Fold 5 RMSE: 1.2480293215439149
LightGBM Fold RMSE: [np.float64(1.2309181643011173), np.float64(1.2325901907249033), np.float64(1.216643647877837), np.float64(1.227986278236269), np.float64(1.2480293215439149)]
LightGBM Mean RMSE: 1.2312335205368083
LightGBM OOF RMSE: 1.2312747344414328


## Deney 13 - XGBoost OOF

In [ ]:
def get_xgb_oof(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            n_estimators=1200,
            learning_rate=0.035,
            max_depth=5,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            tree_method="hist"
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"XGBoost Fold {fold} RMSE:", fold_rmse)

    print("XGBoost Fold RMSE:", fold_scores)
    print("XGBoost Mean RMSE:", np.mean(fold_scores))
    print("XGBoost OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_xgb, test_xgb, scores_xgb = get_xgb_oof(X_fe, y, X_test_fe, cv)

XGBoost Fold 1 RMSE: 1.229662159932237
XGBoost Fold 2 RMSE: 1.231117587938982
XGBoost Fold 3 RMSE: 1.2179742670775981
XGBoost Fold 4 RMSE: 1.2286724371160755
XGBoost Fold 5 RMSE: 1.2481196945798023
XGBoost Fold RMSE: [np.float64(1.229662159932237), np.float64(1.231117587938982), np.float64(1.2179742670775981), np.float64(1.2286724371160755), np.float64(1.2481196945798023)]
XGBoost Mean RMSE: 1.231109229328939
XGBoost OOF RMSE: 1.2311473988025852


## Deney 14 — 4 Model OOF Optimized Blend

In [ ]:
from scipy.optimize import minimize

oof_matrix_4 = np.column_stack([
    oof_cat,
    oof_hgb,
    oof_lgbm,
    oof_xgb
])

test_matrix_4 = np.column_stack([
    test_cat,
    test_hgb,
    test_lgbm,
    test_xgb
])

model_names = [
    "CatBoost",
    "HGB",
    "LightGBM",
    "XGBoost"
]

def blend_rmse_4(weights):
    preds = oof_matrix_4 @ weights
    return rmse(y, preds)

n_models = oof_matrix_4.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_4 = minimize(
    blend_rmse_4,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_4 = result_4.x
best_oof_preds_4 = oof_matrix_4 @ best_weights_4
best_test_preds_4 = test_matrix_4 @ best_weights_4

print("Optimization success:", result_4.success)
print("Best weights:")
for name, weight in zip(model_names, best_weights_4):
    print(f"{name}: {weight:.6f}")

print("4-Model Blend OOF RMSE:", rmse(y, best_oof_preds_4))

Optimization success: True
Best weights:
CatBoost: 0.814445
HGB: 0.117379
LightGBM: 0.068176
XGBoost: 0.000000
4-Model Blend OOF RMSE: 1.2163393057392826


In [ ]:
submission_exp14 = pd.DataFrame({
    "id": test_ids,
    TARGET: best_test_preds_4
})

submission_exp14[TARGET] = submission_exp14[TARGET].clip(0, 10)

print(submission_exp14.shape)
print(submission_exp14.head())
print(submission_exp14[TARGET].describe())

submission_exp14.to_csv("submission_exp14_4model_oof_blend.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   5.991153
1   2                   6.700747
2   3                   3.031903
3   4                   7.185300
4   5                   3.656060
count    24000.000000
mean         5.936518
std          1.860162
min          0.000000
25%          4.652755
50%          6.042710
75%          7.321258
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


## KNN ile Deney

In [ ]:
# ============================================================
# Deney - KNN OOF Regressor
# Tek blok: veri okuma + feature engineering + KNN OOF
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

# ----------------------------
# 1. Veri okuma
# ----------------------------
DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_x.csv")

TARGET = "bilissel_performans_skoru"

X = train.drop(columns=["id", TARGET])
y = train[TARGET]

test_ids = test["id"]
X_test = test.drop(columns=["id"])

print("Train:", X.shape)
print("Test:", X_test.shape)

# ----------------------------
# 2. Feature engineering
# ----------------------------
def prepare_features(X, X_test):
    X_fe = X.copy()
    X_test_fe = X_test.copy()

    log_cols = [
        "uyku_oncesi_kafein_mg",
        "uyku_oncesi_ekran_suresi_dk"
    ]

    for col in log_cols:
        if col in X_fe.columns:
            X_fe[col] = np.log1p(X_fe[col])
            X_test_fe[col] = np.log1p(X_test_fe[col])

    def add_features(df):
        df = df.copy()

        if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

        if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
            df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

        if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
            df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

        if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
            df["dijital_kafein_yuku"] = (
                df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]
            )

        if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
            df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

        return df

    X_fe = add_features(X_fe)
    X_test_fe = add_features(X_test_fe)

    return X_fe, X_test_fe


X_fe, X_test_fe = prepare_features(X, X_test)

print("X_fe:", X_fe.shape)
print("X_test_fe:", X_test_fe.shape)

# ----------------------------
# 3. Ortak RMSE ve CV
# ----------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


cv = KFold(n_splits=5, shuffle=True, random_state=42)

# ----------------------------
# 4. KNN OOF fonksiyonu
# ----------------------------
def get_knn_oof(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    print("Numeric columns:", len(num_cols))
    print("Categorical columns:", len(cat_cols))

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = KNeighborsRegressor(
            n_neighbors=25,
            weights="distance",
            metric="minkowski",
            p=2,
            n_jobs=-1
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"KNN Fold {fold} RMSE:", fold_rmse)

    print("KNN Fold RMSE:", fold_scores)
    print("KNN Mean RMSE:", np.mean(fold_scores))
    print("KNN OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


# ----------------------------
# 5. KNN çalıştır
# ----------------------------
oof_knn, test_knn, scores_knn = get_knn_oof(X_fe, y, X_test_fe, cv)

Train: (56000, 22)
Test: (24000, 22)
X_fe: (56000, 27)
X_test_fe: (24000, 27)
Numeric columns: 20
Categorical columns: 7
KNN Fold 1 RMSE: 1.4457580499194775
KNN Fold 2 RMSE: 1.4265810713502436
KNN Fold 3 RMSE: 1.431239952368049
KNN Fold 4 RMSE: 1.4327170571341614
KNN Fold 5 RMSE: 1.4536697571732202
KNN Fold RMSE: [np.float64(1.4457580499194775), np.float64(1.4265810713502436), np.float64(1.431239952368049), np.float64(1.4327170571341614), np.float64(1.4536697571732202)]
KNN Mean RMSE: 1.4379931775890307
KNN OOF RMSE: 1.438028624333987



## Deney 15 - CatBoost Seed Ensemble

 Mevcut oof_cat / test_cat seed=42 kabul edilerek sadece seed 2024 ve 3407 çalıştırılır.


In [ ]:
from catboost import CatBoostRegressor
import numpy as np

def get_catboost_oof_with_seed(X_fe, y, X_test_fe, cv, seed):
    X_cb = X_fe.copy()
    X_test_cb = X_test_fe.copy()

    cat_cols = X_cb.select_dtypes(include="object").columns.tolist()

    for col in cat_cols:
        X_cb[col] = X_cb[col].fillna("Bilinmiyor").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("Bilinmiyor").astype(str)

    cat_features_idx = [X_cb.columns.get_loc(col) for col in cat_cols]

    oof_preds = np.zeros(len(X_cb))
    test_preds = np.zeros(len(X_test_cb))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
        X_train_fold = X_cb.iloc[train_idx]
        X_val_fold = X_cb.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=1500,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5,
            random_seed=seed,
            verbose=200,
            early_stopping_rounds=100
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_idx,
            eval_set=(X_val_fold, y_val_fold),
            use_best_model=True
        )

        val_pred = model.predict(X_val_fold)
        test_pred = model.predict(X_test_cb)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} - Fold {fold} RMSE:", fold_rmse)

    print(f"Seed {seed} Fold RMSE:", fold_scores)
    print(f"Seed {seed} Mean RMSE:", np.mean(fold_scores))
    print(f"Seed {seed} OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


seeds = [42, 2024, 3407]

cat_oof_list = []
cat_test_list = []
cat_seed_scores = {}

for seed in seeds:
    print("=" * 70)
    print(f"Running CatBoost seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed(
        X_fe,
        y,
        X_test_fe,
        cv,
        seed
    )

    cat_oof_list.append(oof_seed)
    cat_test_list.append(test_seed)

    cat_seed_scores[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_seed_ensemble = np.mean(cat_oof_list, axis=0)
test_cat_seed_ensemble = np.mean(cat_test_list, axis=0)

print("=" * 70)
print("CatBoost Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("CatBoost Seed Ensemble OOF RMSE:", rmse(y, oof_cat_seed_ensemble))

Running CatBoost seed: 42
0:	learn: 2.1899899	test: 2.2002100	best: 2.2002100 (0)	total: 8.09ms	remaining: 12.1s
200:	learn: 1.2203253	test: 1.2387517	best: 1.2387517 (200)	total: 1.49s	remaining: 9.66s
400:	learn: 1.1993460	test: 1.2270202	best: 1.2270202 (400)	total: 2.98s	remaining: 8.16s
600:	learn: 1.1871140	test: 1.2239233	best: 1.2238886 (596)	total: 4.43s	remaining: 6.63s
800:	learn: 1.1768064	test: 1.2228847	best: 1.2228847 (800)	total: 5.94s	remaining: 5.18s
1000:	learn: 1.1675482	test: 1.2223728	best: 1.2223562 (997)	total: 7.4s	remaining: 3.69s
1200:	learn: 1.1580040	test: 1.2223533	best: 1.2222052 (1149)	total: 8.89s	remaining: 2.21s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.222205227
bestIteration = 1149

Shrink model to first 1150 iterations.
Seed 42 - Fold 1 RMSE: 1.2222052278876236
0:	learn: 2.1933248	test: 2.1844791	best: 2.1844791 (0)	total: 7.6ms	remaining: 11.4s
200:	learn: 1.2219388	test: 1.2329316	best: 1.2329316 (200)	total: 1.68s	rema

## Deney 16 - CatBoost Seed Ensemble + HGB + LightGBM OOF Blend

In [ ]:
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

def get_hgb_oof(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.04,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.1,
            random_state=42
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"HGB Fold {fold} RMSE:", fold_rmse)

    print("HGB Mean RMSE:", np.mean(fold_scores))
    print("HGB OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_hgb, test_hgb, scores_hgb = get_hgb_oof(X_fe, y, X_test_fe, cv)

HGB Fold 1 RMSE: 1.228499560987444
HGB Fold 2 RMSE: 1.2290440992060874
HGB Fold 3 RMSE: 1.2159707118361172
HGB Fold 4 RMSE: 1.2243227403073043
HGB Fold 5 RMSE: 1.2450227212554872
HGB Mean RMSE: 1.228571966718488
HGB OOF RMSE: 1.2286084071060954


In [ ]:
import numpy as np

from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

def get_lgbm_oof(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = LGBMRegressor(
            objective="regression",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=25,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.05,
            reg_lambda=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"LightGBM Fold {fold} RMSE:", fold_rmse)

    print("LightGBM Mean RMSE:", np.mean(fold_scores))
    print("LightGBM OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_lgbm, test_lgbm, scores_lgbm = get_lgbm_oof(X_fe, y, X_test_fe, cv)

LightGBM Fold 1 RMSE: 1.2309181643011173
LightGBM Fold 2 RMSE: 1.2325901907249033
LightGBM Fold 3 RMSE: 1.216643647877837
LightGBM Fold 4 RMSE: 1.227986278236269
LightGBM Fold 5 RMSE: 1.2480293215439149
LightGBM Mean RMSE: 1.2312335205368083
LightGBM OOF RMSE: 1.2312747344414328


In [ ]:
from scipy.optimize import minimize
import numpy as np

# Modeller:
# 1. oof_cat_seed_ensemble / test_cat_seed_ensemble
# 2. oof_hgb / test_hgb
# 3. oof_lgbm / test_lgbm

oof_matrix_seed_blend = np.column_stack([
    oof_cat_seed_ensemble,
    oof_hgb,
    oof_lgbm
])

test_matrix_seed_blend = np.column_stack([
    test_cat_seed_ensemble,
    test_hgb,
    test_lgbm
])

model_names_seed_blend = [
    "CatBoost Seed Ensemble",
    "HGB",
    "LightGBM"
]

def blend_rmse_seed(weights):
    preds = oof_matrix_seed_blend @ weights
    return rmse(y, preds)

n_models = oof_matrix_seed_blend.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_seed_blend = minimize(
    blend_rmse_seed,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_seed_blend = result_seed_blend.x
best_oof_preds_seed_blend = oof_matrix_seed_blend @ best_weights_seed_blend
best_test_preds_seed_blend = test_matrix_seed_blend @ best_weights_seed_blend

print("Optimization success:", result_seed_blend.success)
print("Best weights:")

for name, weight in zip(model_names_seed_blend, best_weights_seed_blend):
    print(f"{name}: {weight:.6f}")

print("CatBoost Seed Ensemble OOF RMSE:", rmse(y, oof_cat_seed_ensemble))
print("Seed Ensemble Blend OOF RMSE:", rmse(y, best_oof_preds_seed_blend))

Optimization success: True
Best weights:
CatBoost Seed Ensemble: 0.856283
HGB: 0.118721
LightGBM: 0.024996
CatBoost Seed Ensemble OOF RMSE: 1.216103050186402
Seed Ensemble Blend OOF RMSE: 1.215848349720994


In [ ]:
submission_exp16 = pd.DataFrame({
    "id": test_ids,
    TARGET: best_test_preds_seed_blend
})

submission_exp16[TARGET] = submission_exp16[TARGET].clip(0, 10)

print(submission_exp16.shape)
print(submission_exp16.head())
print(submission_exp16[TARGET].describe())

submission_exp16.to_csv("submission_exp16_seed_ensemble_blend.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   6.000910
1   2                   6.693142
2   3                   3.031077
3   4                   7.161931
4   5                   3.653956
count    24000.000000
mean         5.935599
std          1.859681
min          0.000000
25%          4.651356
50%          6.043203
75%          7.316779
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


# Deney 17 - Feature Engineering v2

In [ ]:
def prepare_features_v2(X, X_test):
    X_fe = X.copy()
    X_test_fe = X_test.copy()

    log_cols = [
        "uyku_oncesi_kafein_mg",
        "uyku_oncesi_ekran_suresi_dk"
    ]

    for col in log_cols:
        if col in X_fe.columns:
            X_fe[col] = np.log1p(X_fe[col])
            X_test_fe[col] = np.log1p(X_test_fe[col])

    def add_features(df):
        df = df.copy()

        # ----------------------------
        # Mevcut başarılı feature'lar
        # ----------------------------

        if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

        if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
            df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

        if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
            df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

        if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
            df["dijital_kafein_yuku"] = (
                df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]
            )

            df["ekran_kafein_toplam_yuku"] = (
                df["uyku_oncesi_ekran_suresi_dk"] + df["uyku_oncesi_kafein_mg"]
            )

        if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
            df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

        # ----------------------------
        # Yeni feature'lar v2
        # ----------------------------

        if "uyku_kalitesi_orani" in df.columns and "stres_skoru" in df.columns:
            df["stres_uyku_kalitesi_orani"] = df["stres_skoru"] / (df["uyku_kalitesi_orani"] + 1)

        if "uyku_kalitesi_orani" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["calisma_uyku_kalitesi_orani"] = df["gunluk_calisma_saati"] / (df["uyku_kalitesi_orani"] + 1)

        if "uyku_bozulma_skoru" in df.columns and "stres_skoru" in df.columns:
            df["stres_uyku_bozulma"] = df["stres_skoru"] * df["uyku_bozulma_skoru"]

        if "gunluk_adim_sayisi" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["adim_calisma_orani"] = df["gunluk_adim_sayisi"] / (df["gunluk_calisma_saati"] + 1)

        if "uyku_kalitesi_orani" in df.columns and "uyku_bozulma_skoru" in df.columns:
            df["uyku_verimlilik_proxy"] = df["uyku_kalitesi_orani"] / (df["uyku_bozulma_skoru"] + 1)

        if "dinlenik_nabiz_bpm" in df.columns and "vucut_kitle_indeksi" in df.columns:
            df["fizyolojik_yuk"] = df["dinlenik_nabiz_bpm"] * df["vucut_kitle_indeksi"]

        if "dinlenik_nabiz_bpm" in df.columns and "gunluk_adim_sayisi" in df.columns:
            df["nabiz_aktivite_orani"] = df["dinlenik_nabiz_bpm"] / (df["gunluk_adim_sayisi"] + 1)

        return df

    X_fe = add_features(X_fe)
    X_test_fe = add_features(X_test_fe)

    return X_fe, X_test_fe


X_fe_v2, X_test_fe_v2 = prepare_features_v2(X, X_test)

print("X_fe_v2:", X_fe_v2.shape)
print("X_test_fe_v2:", X_test_fe_v2.shape)

X_fe_v2: (56000, 35)
X_test_fe_v2: (24000, 35)


# Deney 17A - CatBoost Seed Ensemble with Feature Engineering v2

In [ ]:
seeds = [42, 2024, 3407]

cat_oof_list_v2 = []
cat_test_list_v2 = []
cat_seed_scores_v2 = {}

for seed in seeds:
    print("=" * 70)
    print(f"Running CatBoost v2 seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed(
        X_fe_v2,
        y,
        X_test_fe_v2,
        cv,
        seed
    )

    cat_oof_list_v2.append(oof_seed)
    cat_test_list_v2.append(test_seed)

    cat_seed_scores_v2[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_seed_ensemble_v2 = np.mean(cat_oof_list_v2, axis=0)
test_cat_seed_ensemble_v2 = np.mean(cat_test_list_v2, axis=0)

print("=" * 70)
print("CatBoost v2 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_v2.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("CatBoost v2 Seed Ensemble OOF RMSE:", rmse(y, oof_cat_seed_ensemble_v2))

Running CatBoost v2 seed: 42
0:	learn: 2.1865986	test: 2.1976741	best: 2.1976741 (0)	total: 7.91ms	remaining: 11.9s
200:	learn: 1.2220695	test: 1.2389752	best: 1.2389752 (200)	total: 1.58s	remaining: 10.2s
400:	learn: 1.1996527	test: 1.2264201	best: 1.2264201 (400)	total: 3.17s	remaining: 8.69s
600:	learn: 1.1862719	test: 1.2236896	best: 1.2236811 (598)	total: 4.73s	remaining: 7.08s
800:	learn: 1.1753029	test: 1.2223126	best: 1.2223082 (796)	total: 6.27s	remaining: 5.47s
1000:	learn: 1.1639281	test: 1.2219127	best: 1.2218106 (955)	total: 7.93s	remaining: 3.95s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.221712751
bestIteration = 1058

Shrink model to first 1059 iterations.
Seed 42 - Fold 1 RMSE: 1.2217127517226791
0:	learn: 2.1909521	test: 2.1823133	best: 2.1823133 (0)	total: 8.25ms	remaining: 12.4s
200:	learn: 1.2242263	test: 1.2340560	best: 1.2340560 (200)	total: 1.66s	remaining: 10.7s
400:	learn: 1.2005147	test: 1.2218411	best: 1.2218411 (400)	total: 3.3s	re

# Deney 17B - HGB and LightGBM OOF with Feature Engineering v2

In [ ]:
oof_hgb_v2, test_hgb_v2, scores_hgb_v2 = get_hgb_oof(
    X_fe_v2,
    y,
    X_test_fe_v2,
    cv
)

oof_lgbm_v2, test_lgbm_v2, scores_lgbm_v2 = get_lgbm_oof(
    X_fe_v2,
    y,
    X_test_fe_v2,
    cv
)

HGB Fold 1 RMSE: 1.2310989337647074
HGB Fold 2 RMSE: 1.2326233657114194
HGB Fold 3 RMSE: 1.2169084824803957
HGB Fold 4 RMSE: 1.2250622629721526
HGB Fold 5 RMSE: 1.2443529842307368
HGB Mean RMSE: 1.2300092058318826
HGB OOF RMSE: 1.2300425276119096
LightGBM Fold 1 RMSE: 1.2324687563371692
LightGBM Fold 2 RMSE: 1.2334375406247986
LightGBM Fold 3 RMSE: 1.2173729895644247
LightGBM Fold 4 RMSE: 1.224969754798498
LightGBM Fold 5 RMSE: 1.2474545745352474
LightGBM Mean RMSE: 1.2311407231720275
LightGBM OOF RMSE: 1.2311814012070579


# Deney 17C - OOF Optimized Blend with Feature Engineering v2

In [ ]:
from scipy.optimize import minimize

oof_matrix_v2 = np.column_stack([
    oof_cat_seed_ensemble_v2,
    oof_hgb_v2,
    oof_lgbm_v2
])

test_matrix_v2 = np.column_stack([
    test_cat_seed_ensemble_v2,
    test_hgb_v2,
    test_lgbm_v2
])

model_names_v2 = [
    "CatBoost Seed Ensemble v2",
    "HGB v2",
    "LightGBM v2"
]

def blend_rmse_v2(weights):
    preds = oof_matrix_v2 @ weights
    return rmse(y, preds)

n_models = oof_matrix_v2.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_v2 = minimize(
    blend_rmse_v2,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_v2 = result_v2.x
best_oof_preds_v2 = oof_matrix_v2 @ best_weights_v2
best_test_preds_v2 = test_matrix_v2 @ best_weights_v2

print("Optimization success:", result_v2.success)
print("Best weights:")

for name, weight in zip(model_names_v2, best_weights_v2):
    print(f"{name}: {weight:.6f}")

print("Feature v2 Blend OOF RMSE:", rmse(y, best_oof_preds_v2))

Optimization success: True
Best weights:
CatBoost Seed Ensemble v2: 0.858733
HGB v2: 0.078906
LightGBM v2: 0.062361
Feature v2 Blend OOF RMSE: 1.216549505068112


# Deney 18 - CatBoost v1 Feature Set ile 5 Seed Ensemble

In [ ]:
from catboost import CatBoostRegressor
import numpy as np

def get_catboost_oof_with_seed(X_fe, y, X_test_fe, cv, seed):
    X_cb = X_fe.copy()
    X_test_cb = X_test_fe.copy()

    cat_cols = X_cb.select_dtypes(include="object").columns.tolist()

    for col in cat_cols:
        X_cb[col] = X_cb[col].fillna("Bilinmiyor").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("Bilinmiyor").astype(str)

    cat_features_idx = [X_cb.columns.get_loc(col) for col in cat_cols]

    oof_preds = np.zeros(len(X_cb))
    test_preds = np.zeros(len(X_test_cb))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
        X_train_fold = X_cb.iloc[train_idx]
        X_val_fold = X_cb.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=1500,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5,
            random_seed=seed,
            verbose=200,
            early_stopping_rounds=100
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_idx,
            eval_set=(X_val_fold, y_val_fold),
            use_best_model=True
        )

        val_pred = model.predict(X_val_fold)
        test_pred = model.predict(X_test_cb)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} - Fold {fold} RMSE:", fold_rmse)

    print(f"Seed {seed} Mean RMSE:", np.mean(fold_scores))
    print(f"Seed {seed} OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


# v1 feature set ile 5 seed çalıştırıyoruz
seeds_v1_5 = [42, 2024, 3407, 777, 999]

cat_oof_list_5 = []
cat_test_list_5 = []
cat_seed_scores_5 = {}

for seed in seeds_v1_5:
    print("=" * 70)
    print(f"Running CatBoost v1 seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed(
        X_fe,
        y,
        X_test_fe,
        cv,
        seed
    )

    cat_oof_list_5.append(oof_seed)
    cat_test_list_5.append(test_seed)

    cat_seed_scores_5[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_seed_ensemble_5 = np.mean(cat_oof_list_5, axis=0)
test_cat_seed_ensemble_5 = np.mean(cat_test_list_5, axis=0)

print("=" * 70)
print("CatBoost 5 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_5.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("CatBoost 3 Seed Ensemble OOF RMSE eski değer: 1.216103")
print("CatBoost 5 Seed Ensemble OOF RMSE:", rmse(y, oof_cat_seed_ensemble_5))

Running CatBoost v1 seed: 42
0:	learn: 2.1899899	test: 2.2002100	best: 2.2002100 (0)	total: 8.78ms	remaining: 13.2s
200:	learn: 1.2203253	test: 1.2387517	best: 1.2387517 (200)	total: 1.7s	remaining: 11s
400:	learn: 1.1993460	test: 1.2270202	best: 1.2270202 (400)	total: 3.17s	remaining: 8.7s
600:	learn: 1.1871140	test: 1.2239233	best: 1.2238886 (596)	total: 4.69s	remaining: 7.01s
800:	learn: 1.1768064	test: 1.2228847	best: 1.2228847 (800)	total: 6.16s	remaining: 5.38s
1000:	learn: 1.1675482	test: 1.2223728	best: 1.2223562 (997)	total: 7.66s	remaining: 3.82s
1200:	learn: 1.1580040	test: 1.2223533	best: 1.2222052 (1149)	total: 9.13s	remaining: 2.27s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.222205227
bestIteration = 1149

Shrink model to first 1150 iterations.
Seed 42 - Fold 1 RMSE: 1.2222052278876236
0:	learn: 2.1933248	test: 2.1844791	best: 2.1844791 (0)	total: 7.19ms	remaining: 10.8s
200:	learn: 1.2219388	test: 1.2329316	best: 1.2329316 (200)	total: 1.74s	rem

# Deney 19 - 5 Seed CatBoost + HGB + LightGBM OOF Blend

In [ ]:
from scipy.optimize import minimize
import numpy as np

oof_matrix_5seed_blend = np.column_stack([
    oof_cat_seed_ensemble_5,
    oof_hgb,
    oof_lgbm
])

test_matrix_5seed_blend = np.column_stack([
    test_cat_seed_ensemble_5,
    test_hgb,
    test_lgbm
])

model_names_5seed_blend = [
    "CatBoost 5 Seed Ensemble",
    "HGB",
    "LightGBM"
]

def blend_rmse_5seed(weights):
    preds = oof_matrix_5seed_blend @ weights
    return rmse(y, preds)

n_models = oof_matrix_5seed_blend.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_5seed_blend = minimize(
    blend_rmse_5seed,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_5seed_blend = result_5seed_blend.x
best_oof_preds_5seed_blend = oof_matrix_5seed_blend @ best_weights_5seed_blend
best_test_preds_5seed_blend = test_matrix_5seed_blend @ best_weights_5seed_blend

print("Optimization success:", result_5seed_blend.success)
print("Best weights:")

for name, weight in zip(model_names_5seed_blend, best_weights_5seed_blend):
    print(f"{name}: {weight:.6f}")

print("Deney 16 OOF RMSE eski en iyi: 1.215848")
print("5 Seed Blend OOF RMSE:", rmse(y, best_oof_preds_5seed_blend))

Optimization success: True
Best weights:
CatBoost 5 Seed Ensemble: 0.862233
HGB: 0.113767
LightGBM: 0.024001
Deney 16 OOF RMSE eski en iyi: 1.215848
5 Seed Blend OOF RMSE: 1.2157831137897666


In [ ]:
submission_exp19 = pd.DataFrame({
    "id": test_ids,
    TARGET: best_test_preds_5seed_blend
})

submission_exp19[TARGET] = submission_exp19[TARGET].clip(0, 10)

print(submission_exp19.shape)
print(submission_exp19.head())
print(submission_exp19[TARGET].describe())

submission_exp19.to_csv("submission_exp19_5seed_blend.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   6.004932
1   2                   6.703876
2   3                   3.025952
3   4                   7.165925
4   5                   3.647632
count    24000.000000
mean         5.934777
std          1.859313
min          0.000000
25%          4.650256
50%          6.042210
75%          7.316993
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


# Deney 20 - Missing Indicators + Categorical Interactions

In [14]:
def prepare_features_v3(X, X_test):
    X_fe = X.copy()
    X_test_fe = X_test.copy()

    # --------------------------------------------------------
    # 1. Missing indicator'ları ekle
    # Train ve test için aynı kolonlarda indicator üretelim
    # --------------------------------------------------------
    all_cols = X_fe.columns.tolist()

    for col in all_cols:
        if X_fe[col].isnull().any() or X_test_fe[col].isnull().any():
            X_fe[f"{col}_missing"] = X_fe[col].isnull().astype(int)
            X_test_fe[f"{col}_missing"] = X_test_fe[col].isnull().astype(int)

    # --------------------------------------------------------
    # 2. Log transform - v1'de işe yarayan dönüşüm
    # --------------------------------------------------------
    log_cols = [
        "uyku_oncesi_kafein_mg",
        "uyku_oncesi_ekran_suresi_dk"
    ]

    for col in log_cols:
        if col in X_fe.columns:
            X_fe[col] = np.log1p(X_fe[col])
            X_test_fe[col] = np.log1p(X_test_fe[col])

    def add_features(df):
        df = df.copy()

        # ----------------------------------------------------
        # v1'de işe yarayan feature'lar
        # ----------------------------------------------------
        if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

        if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
            df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

        if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
            df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

        if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
            df["dijital_kafein_yuku"] = (
                df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]
            )

        if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
            df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

        # ----------------------------------------------------
        # Yeni: kategorik interaction feature'lar
        # CatBoost bunları native categorical olarak iyi kullanabilir
        # ----------------------------------------------------
        def safe_str_col(c):
            return df[c].fillna("Bilinmiyor").astype(str)

        if "meslek" in df.columns and "kronotip" in df.columns:
            df["meslek_kronotip"] = safe_str_col("meslek") + "_" + safe_str_col("kronotip")

        if "gun_tipi" in df.columns and "kronotip" in df.columns:
            df["gun_tipi_kronotip"] = safe_str_col("gun_tipi") + "_" + safe_str_col("kronotip")

        if "ruh_sagligi_durumu" in df.columns and "meslek" in df.columns:
            df["ruh_sagligi_meslek"] = safe_str_col("ruh_sagligi_durumu") + "_" + safe_str_col("meslek")

        if "cinsiyet" in df.columns and "kronotip" in df.columns:
            df["cinsiyet_kronotip"] = safe_str_col("cinsiyet") + "_" + safe_str_col("kronotip")

        return df

    X_fe = add_features(X_fe)
    X_test_fe = add_features(X_test_fe)

    return X_fe, X_test_fe


X_fe_v3, X_test_fe_v3 = prepare_features_v3(X, X_test)

print("X_fe_v3:", X_fe_v3.shape)
print("X_test_fe_v3:", X_test_fe_v3.shape)
print("New columns added:", X_fe_v3.shape[1] - X.shape[1])

X_fe_v3: (56000, 35)
X_test_fe_v3: (24000, 35)
New columns added: 12


In [15]:
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

def get_hgb_oof_dense(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ],
        sparse_threshold=0
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.04,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.1,
            random_state=42
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"HGB Fold {fold} RMSE:", fold_rmse)

    print("HGB Mean RMSE:", np.mean(fold_scores))
    print("HGB OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores

# Deney 20A - CatBoost 3 Seed Ensemble with Feature v3

In [16]:
# CatBoost v3 kısmı aynı kalacak
seeds_v3 = [42, 2024, 3407]

cat_oof_list_v3 = []
cat_test_list_v3 = []
cat_seed_scores_v3 = {}

for seed in seeds_v3:
    print("=" * 70)
    print(f"Running CatBoost v3 seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed(
        X_fe_v3,
        y,
        X_test_fe_v3,
        cv,
        seed
    )

    cat_oof_list_v3.append(oof_seed)
    cat_test_list_v3.append(test_seed)

    cat_seed_scores_v3[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_seed_ensemble_v3 = np.mean(cat_oof_list_v3, axis=0)
test_cat_seed_ensemble_v3 = np.mean(cat_test_list_v3, axis=0)

print("=" * 70)
print("CatBoost v3 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_v3.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("Old v1 3-seed CatBoost OOF: 1.216103")
print("CatBoost v3 3-Seed Ensemble OOF RMSE:", rmse(y, oof_cat_seed_ensemble_v3))


# BURASI DEĞİŞTİ: get_hgb_oof yerine get_hgb_oof_dense
oof_hgb_v3, test_hgb_v3, scores_hgb_v3 = get_hgb_oof_dense(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

# LightGBM aynı kalabilir
oof_lgbm_v3, test_lgbm_v3, scores_lgbm_v3 = get_lgbm_oof(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

Running CatBoost v3 seed: 42


NameError: name 'get_catboost_oof_with_seed' is not defined

# Deney 20B - HGB and LightGBM with Feature v3

In [17]:
oof_hgb_v3, test_hgb_v3, scores_hgb_v3 = get_hgb_oof_dense(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

oof_lgbm_v3, test_lgbm_v3, scores_lgbm_v3 = get_lgbm_oof(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

HGB Fold 1 RMSE: 1.2307257608746391
HGB Fold 2 RMSE: 1.2279666280094808
HGB Fold 3 RMSE: 1.2180883161193514
HGB Fold 4 RMSE: 1.2249099451212055
HGB Fold 5 RMSE: 1.2450940604787741
HGB Mean RMSE: 1.22935694212069
HGB OOF RMSE: 1.2293893343402866


NameError: name 'get_lgbm_oof' is not defined

# Deney 20C - OOF Optimized Blend with Feature v3

In [ ]:
from scipy.optimize import minimize

oof_matrix_v3 = np.column_stack([
    oof_cat_seed_ensemble_v3,
    oof_hgb_v3,
    oof_lgbm_v3
])

test_matrix_v3 = np.column_stack([
    test_cat_seed_ensemble_v3,
    test_hgb_v3,
    test_lgbm_v3
])

model_names_v3 = [
    "CatBoost 3 Seed Ensemble v3",
    "HGB v3",
    "LightGBM v3"
]

def blend_rmse_v3(weights):
    preds = oof_matrix_v3 @ weights
    return rmse(y, preds)

n_models = oof_matrix_v3.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_v3 = minimize(
    blend_rmse_v3,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_v3 = result_v3.x
best_oof_preds_v3 = oof_matrix_v3 @ best_weights_v3
best_test_preds_v3 = test_matrix_v3 @ best_weights_v3

print("Optimization success:", result_v3.success)
print("Best weights:")

for name, weight in zip(model_names_v3, best_weights_v3):
    print(f"{name}: {weight:.6f}")

print("Feature v3 Blend OOF RMSE:", rmse(y, best_oof_preds_v3))

Optimization success: True
Best weights:
CatBoost 3 Seed Ensemble v3: 0.871066
HGB v3: 0.089645
LightGBM v3: 0.039289
Feature v3 Blend OOF RMSE: 1.215493973927574


# Deney 20D - CatBoost v3 5 Seed Ensemble

In [18]:
seeds_v3_5 = [42, 2024, 3407, 777, 999]

cat_oof_list_v3_5 = []
cat_test_list_v3_5 = []
cat_seed_scores_v3_5 = {}

for seed in seeds_v3_5:
    print("=" * 70)
    print(f"Running CatBoost v3 seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed(
        X_fe_v3,
        y,
        X_test_fe_v3,
        cv,
        seed
    )

    cat_oof_list_v3_5.append(oof_seed)
    cat_test_list_v3_5.append(test_seed)

    cat_seed_scores_v3_5[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_seed_ensemble_v3_5 = np.mean(cat_oof_list_v3_5, axis=0)
test_cat_seed_ensemble_v3_5 = np.mean(cat_test_list_v3_5, axis=0)

print("=" * 70)
print("CatBoost v3 5 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_v3_5.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("CatBoost v3 3-Seed Ensemble OOF eski:", rmse(y, oof_cat_seed_ensemble_v3))
print("CatBoost v3 5-Seed Ensemble OOF RMSE:", rmse(y, oof_cat_seed_ensemble_v3_5))

Running CatBoost v3 seed: 42


NameError: name 'get_catboost_oof_with_seed' is not defined

# Deney 20E - v3 5 Seed CatBoost + HGB v3 + LightGBM v3 Blend

In [19]:
from scipy.optimize import minimize

oof_matrix_v3_5 = np.column_stack([
    oof_cat_seed_ensemble_v3_5,
    oof_hgb_v3,
    oof_lgbm_v3
])

test_matrix_v3_5 = np.column_stack([
    test_cat_seed_ensemble_v3_5,
    test_hgb_v3,
    test_lgbm_v3
])

model_names_v3_5 = [
    "CatBoost v3 5 Seed Ensemble",
    "HGB v3",
    "LightGBM v3"
]

def blend_rmse_v3_5(weights):
    preds = oof_matrix_v3_5 @ weights
    return rmse(y, preds)

n_models = oof_matrix_v3_5.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_v3_5 = minimize(
    blend_rmse_v3_5,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_v3_5 = result_v3_5.x
best_oof_preds_v3_5 = oof_matrix_v3_5 @ best_weights_v3_5
best_test_preds_v3_5 = test_matrix_v3_5 @ best_weights_v3_5

print("Optimization success:", result_v3_5.success)
print("Best weights:")

for name, weight in zip(model_names_v3_5, best_weights_v3_5):
    print(f"{name}: {weight:.6f}")

print("Deney 20C v3 3-seed blend OOF:", rmse(y, best_oof_preds_v3))
print("Deney 20E v3 5-seed blend OOF:", rmse(y, best_oof_preds_v3_5))

NameError: name 'oof_cat_seed_ensemble_v3_5' is not defined

In [ ]:
submission_exp20e = pd.DataFrame({
    "id": test_ids,
    TARGET: best_test_preds_v3_5
})

submission_exp20e[TARGET] = submission_exp20e[TARGET].clip(0, 10)

print(submission_exp20e.shape)
print(submission_exp20e.head())
print(submission_exp20e[TARGET].describe())

submission_exp20e.to_csv("submission_exp20e_v3_5seed_blend.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   5.995484
1   2                   6.766010
2   3                   3.037712
3   4                   7.159019
4   5                   3.628314
count    24000.000000
mean         5.936977
std          1.862729
min          0.000000
25%          4.641429
50%          6.044348
75%          7.327849
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


# Deney 21 - CatBoost v3 + Numeric Median Imputation

In [ ]:
from catboost import CatBoostRegressor
import numpy as np

def get_catboost_oof_with_seed_median(X_fe, y, X_test_fe, cv, seed):
    X_cb = X_fe.copy()
    X_test_cb = X_test_fe.copy()

    cat_cols = X_cb.select_dtypes(include="object").columns.tolist()
    num_cols = X_cb.select_dtypes(include=np.number).columns.tolist()

    # Categorical NaN -> Bilinmiyor
    for col in cat_cols:
        X_cb[col] = X_cb[col].fillna("Bilinmiyor").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("Bilinmiyor").astype(str)

    cat_features_idx = [X_cb.columns.get_loc(col) for col in cat_cols]

    oof_preds = np.zeros(len(X_cb))
    test_preds = np.zeros(len(X_test_cb))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
        X_train_fold = X_cb.iloc[train_idx].copy()
        X_val_fold = X_cb.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        X_test_fold = X_test_cb.copy()

        # Numeric median sadece train fold üzerinden öğreniliyor
        medians = X_train_fold[num_cols].median()

        X_train_fold[num_cols] = X_train_fold[num_cols].fillna(medians)
        X_val_fold[num_cols] = X_val_fold[num_cols].fillna(medians)
        X_test_fold[num_cols] = X_test_fold[num_cols].fillna(medians)

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=1500,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5,
            random_seed=seed,
            verbose=200,
            early_stopping_rounds=100
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_idx,
            eval_set=(X_val_fold, y_val_fold),
            use_best_model=True
        )

        val_pred = model.predict(X_val_fold)
        test_pred = model.predict(X_test_fold)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} - Fold {fold} RMSE:", fold_rmse)

    print(f"Seed {seed} Mean RMSE:", np.mean(fold_scores))
    print(f"Seed {seed} OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores

# Deney 21A - CatBoost v3 + Median Imputation - 3 Seed

In [ ]:
seeds_median_3 = [42, 2024, 3407]

cat_oof_list_median_3 = []
cat_test_list_median_3 = []
cat_seed_scores_median_3 = {}

for seed in seeds_median_3:
    print("=" * 70)
    print(f"Running CatBoost v3 median seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed_median(
        X_fe_v3,
        y,
        X_test_fe_v3,
        cv,
        seed
    )

    cat_oof_list_median_3.append(oof_seed)
    cat_test_list_median_3.append(test_seed)

    cat_seed_scores_median_3[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_median_3 = np.mean(cat_oof_list_median_3, axis=0)
test_cat_median_3 = np.mean(cat_test_list_median_3, axis=0)

print("=" * 70)
print("CatBoost v3 Median 3 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_median_3.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("Eski v3 3-seed CatBoost OOF:", rmse(y, oof_cat_seed_ensemble_v3))
print("Median v3 3-seed CatBoost OOF:", rmse(y, oof_cat_median_3))

Running CatBoost v3 median seed: 42
0:	learn: 2.1884436	test: 2.1992729	best: 2.1992729 (0)	total: 9.24ms	remaining: 13.9s
200:	learn: 1.2209683	test: 1.2374999	best: 1.2374999 (200)	total: 2.63s	remaining: 17s
400:	learn: 1.2004931	test: 1.2257634	best: 1.2257634 (400)	total: 4.77s	remaining: 13.1s
600:	learn: 1.1892405	test: 1.2228999	best: 1.2228999 (600)	total: 6.92s	remaining: 10.4s
800:	learn: 1.1801532	test: 1.2219336	best: 1.2218889 (775)	total: 9.1s	remaining: 7.94s
1000:	learn: 1.1723566	test: 1.2214341	best: 1.2214273 (997)	total: 11.5s	remaining: 5.72s
1200:	learn: 1.1635280	test: 1.2211876	best: 1.2211876 (1199)	total: 13.7s	remaining: 3.42s
1400:	learn: 1.1553360	test: 1.2209921	best: 1.2209284 (1338)	total: 16s	remaining: 1.13s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.220928352
bestIteration = 1338

Shrink model to first 1339 iterations.
Seed 42 - Fold 1 RMSE: 1.220928353241764
0:	learn: 2.1924236	test: 2.1834872	best: 2.1834872 (0)	total: 11.

# Deney 21B - Median CatBoost v3 3 Seed + HGB v3 + LGBM v3 Blend

In [ ]:
from scipy.optimize import minimize

oof_matrix_median_3 = np.column_stack([
    oof_cat_median_3,
    oof_hgb_v3,
    oof_lgbm_v3
])

test_matrix_median_3 = np.column_stack([
    test_cat_median_3,
    test_hgb_v3,
    test_lgbm_v3
])

model_names_median_3 = [
    "CatBoost v3 Median 3 Seed",
    "HGB v3",
    "LightGBM v3"
]

def blend_rmse_median_3(weights):
    preds = oof_matrix_median_3 @ weights
    return rmse(y, preds)

n_models = oof_matrix_median_3.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_median_3 = minimize(
    blend_rmse_median_3,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_median_3 = result_median_3.x
best_oof_preds_median_3 = oof_matrix_median_3 @ best_weights_median_3
best_test_preds_median_3 = test_matrix_median_3 @ best_weights_median_3

print("Optimization success:", result_median_3.success)
print("Best weights:")

for name, weight in zip(model_names_median_3, best_weights_median_3):
    print(f"{name}: {weight:.6f}")

print("Eski en iyi Deney 20E OOF:", rmse(y, best_oof_preds_v3_5))
print("Median 3-seed blend OOF:", rmse(y, best_oof_preds_median_3))

Optimization success: True
Best weights:
CatBoost v3 Median 3 Seed: 0.910760
HGB v3: 0.070544
LightGBM v3: 0.018697
Eski en iyi Deney 20E OOF: 1.2153820986053874
Median 3-seed blend OOF: 1.2152609001670303


# Deney 21C - CatBoost v3 + Median Imputation - 5 Seed

In [ ]:
seeds_median_5 = [42, 2024, 3407, 777, 999]

cat_oof_list_median_5 = []
cat_test_list_median_5 = []
cat_seed_scores_median_5 = {}

for seed in seeds_median_5:
    print("=" * 70)
    print(f"Running CatBoost v3 median seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed_median(
        X_fe_v3,
        y,
        X_test_fe_v3,
        cv,
        seed
    )

    cat_oof_list_median_5.append(oof_seed)
    cat_test_list_median_5.append(test_seed)

    cat_seed_scores_median_5[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_median_5 = np.mean(cat_oof_list_median_5, axis=0)
test_cat_median_5 = np.mean(cat_test_list_median_5, axis=0)

print("=" * 70)
print("CatBoost v3 Median 5 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_median_5.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("Median v3 3-seed CatBoost OOF:", rmse(y, oof_cat_median_3))
print("Median v3 5-seed CatBoost OOF:", rmse(y, oof_cat_median_5))

Running CatBoost v3 median seed: 42
0:	learn: 2.1884436	test: 2.1992729	best: 2.1992729 (0)	total: 10.9ms	remaining: 16.3s
200:	learn: 1.2209683	test: 1.2374999	best: 1.2374999 (200)	total: 2.18s	remaining: 14.1s
400:	learn: 1.2004931	test: 1.2257634	best: 1.2257634 (400)	total: 4.35s	remaining: 11.9s
600:	learn: 1.1892405	test: 1.2228999	best: 1.2228999 (600)	total: 6.54s	remaining: 9.79s
800:	learn: 1.1801532	test: 1.2219336	best: 1.2218889 (775)	total: 8.71s	remaining: 7.61s
1000:	learn: 1.1723566	test: 1.2214341	best: 1.2214273 (997)	total: 10.9s	remaining: 5.45s
1200:	learn: 1.1635280	test: 1.2211876	best: 1.2211876 (1199)	total: 13.2s	remaining: 3.28s
1400:	learn: 1.1553360	test: 1.2209921	best: 1.2209284 (1338)	total: 15.4s	remaining: 1.09s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.220928352
bestIteration = 1338

Shrink model to first 1339 iterations.
Seed 42 - Fold 1 RMSE: 1.220928353241764
0:	learn: 2.1924236	test: 2.1834872	best: 2.1834872 (0)	total

# Deney 21D - Median CatBoost v3 5 Seed + HGB v3 + LGBM v3 Blend

In [ ]:
from scipy.optimize import minimize

oof_matrix_median_5 = np.column_stack([
    oof_cat_median_5,
    oof_hgb_v3,
    oof_lgbm_v3
])

test_matrix_median_5 = np.column_stack([
    test_cat_median_5,
    test_hgb_v3,
    test_lgbm_v3
])

model_names_median_5 = [
    "CatBoost v3 Median 5 Seed",
    "HGB v3",
    "LightGBM v3"
]

def blend_rmse_median_5(weights):
    preds = oof_matrix_median_5 @ weights
    return rmse(y, preds)

n_models = oof_matrix_median_5.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_median_5 = minimize(
    blend_rmse_median_5,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_median_5 = result_median_5.x
best_oof_preds_median_5 = oof_matrix_median_5 @ best_weights_median_5
best_test_preds_median_5 = test_matrix_median_5 @ best_weights_median_5

print("Optimization success:", result_median_5.success)
print("Best weights:")

for name, weight in zip(model_names_median_5, best_weights_median_5):
    print(f"{name}: {weight:.6f}")

print("Eski en iyi Median 3-seed blend OOF:", rmse(y, best_oof_preds_median_3))
print("Median 5-seed blend OOF:", rmse(y, best_oof_preds_median_5))


Optimization success: True
Best weights:
CatBoost v3 Median 5 Seed: 0.918478
HGB v3: 0.066655
LightGBM v3: 0.014867
Eski en iyi Median 3-seed blend OOF: 1.2152609001670303
Median 5-seed blend OOF: 1.215180348002664


# Deney 22 - Feature v4
# Feature v3 + group-based relative numeric features

In [ ]:
def prepare_features_v4_group_relative(X, X_test):
    # Önce mevcut başarılı v3 feature setini üret
    X_fe, X_test_fe = prepare_features_v3(X, X_test)

    def add_group_relative_feature(train_df, test_df, group_col, num_col):
        if group_col not in train_df.columns or num_col not in train_df.columns:
            return train_df, test_df

        train_df = train_df.copy()
        test_df = test_df.copy()

        train_group = train_df[group_col].fillna("Bilinmiyor").astype(str)
        test_group = test_df[group_col].fillna("Bilinmiyor").astype(str)

        global_median = train_df[num_col].median()

        group_mean = train_df.groupby(train_group)[num_col].mean()

        train_group_mean = train_group.map(group_mean).fillna(global_median)
        test_group_mean = test_group.map(group_mean).fillna(global_median)

        base_name = f"{num_col}_rel_{group_col}"

        train_df[f"{base_name}_diff"] = train_df[num_col] - train_group_mean
        test_df[f"{base_name}_diff"] = test_df[num_col] - test_group_mean

        train_df[f"{base_name}_ratio"] = train_df[num_col] / (train_group_mean + 1)
        test_df[f"{base_name}_ratio"] = test_df[num_col] / (test_group_mean + 1)

        return train_df, test_df

    # Kontrollü kombinasyonlar
    group_numeric_pairs = [
        ("meslek", "stres_skoru"),
        ("meslek", "gunluk_adim_sayisi"),
        ("meslek", "gunluk_calisma_saati"),

        ("kronotip", "uyku_oncesi_kafein_mg"),
        ("kronotip", "uyku_oncesi_ekran_suresi_dk"),
        ("kronotip", "uyku_kalitesi_orani"),
        ("kronotip", "uyku_bozulma_skoru"),

        ("gun_tipi", "gunluk_calisma_saati"),
        ("gun_tipi", "uyku_oncesi_kafein_mg"),
        ("gun_tipi", "uyku_oncesi_ekran_suresi_dk"),

        ("meslek_kronotip", "stres_skoru"),
        ("meslek_kronotip", "gunluk_adim_sayisi"),
        ("meslek_kronotip", "gunluk_calisma_saati"),
    ]

    for group_col, num_col in group_numeric_pairs:
        X_fe, X_test_fe = add_group_relative_feature(
            X_fe,
            X_test_fe,
            group_col,
            num_col
        )

    return X_fe, X_test_fe


X_fe_v4, X_test_fe_v4 = prepare_features_v4_group_relative(X, X_test)

print("X_fe_v4:", X_fe_v4.shape)
print("X_test_fe_v4:", X_test_fe_v4.shape)
print("New columns vs original:", X_fe_v4.shape[1] - X.shape[1])

X_fe_v4: (56000, 63)
X_test_fe_v4: (24000, 63)
New columns vs original: 41


# Deney 22A - CatBoost v4 + Median Imputation - 3 Seed

In [ ]:
seeds_v4_median_3 = [42, 2024, 3407]

cat_oof_list_v4_median_3 = []
cat_test_list_v4_median_3 = []
cat_seed_scores_v4_median_3 = {}

for seed in seeds_v4_median_3:
    print("=" * 70)
    print(f"Running CatBoost v4 median seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed_median(
        X_fe_v4,
        y,
        X_test_fe_v4,
        cv,
        seed
    )

    cat_oof_list_v4_median_3.append(oof_seed)
    cat_test_list_v4_median_3.append(test_seed)

    cat_seed_scores_v4_median_3[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_v4_median_3 = np.mean(cat_oof_list_v4_median_3, axis=0)
test_cat_v4_median_3 = np.mean(cat_test_list_v4_median_3, axis=0)

print("=" * 70)
print("CatBoost v4 Median 3 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_v4_median_3.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("Eski v3 Median 3-seed CatBoost OOF:", rmse(y, oof_cat_median_3))
print("v4 Median 3-seed CatBoost OOF:", rmse(y, oof_cat_v4_median_3))

Running CatBoost v4 median seed: 42
0:	learn: 2.1874944	test: 2.1985876	best: 2.1985876 (0)	total: 10.9ms	remaining: 16.3s
200:	learn: 1.2212564	test: 1.2377082	best: 1.2377082 (200)	total: 2.84s	remaining: 18.4s
400:	learn: 1.1999816	test: 1.2261461	best: 1.2261461 (400)	total: 5.36s	remaining: 14.7s
600:	learn: 1.1877835	test: 1.2228349	best: 1.2227841 (583)	total: 7.7s	remaining: 11.5s
800:	learn: 1.1776386	test: 1.2219384	best: 1.2219384 (800)	total: 10s	remaining: 8.74s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.221799436
bestIteration = 829

Shrink model to first 830 iterations.
Seed 42 - Fold 1 RMSE: 1.221799436911277
0:	learn: 2.1914467	test: 2.1824660	best: 2.1824660 (0)	total: 10.9ms	remaining: 16.4s
200:	learn: 1.2215351	test: 1.2305462	best: 1.2305462 (200)	total: 2.34s	remaining: 15.1s
400:	learn: 1.2008059	test: 1.2198962	best: 1.2198946 (398)	total: 4.65s	remaining: 12.7s
600:	learn: 1.1878002	test: 1.2174478	best: 1.2174419 (598)	total: 6.94s	r

# Deney 22B - HGB and LightGBM with Feature v4

In [ ]:
oof_hgb_v4, test_hgb_v4, scores_hgb_v4 = get_hgb_oof_dense(
    X_fe_v4,
    y,
    X_test_fe_v4,
    cv
)

oof_lgbm_v4, test_lgbm_v4, scores_lgbm_v4 = get_lgbm_oof(
    X_fe_v4,
    y,
    X_test_fe_v4,
    cv
)

HGB Fold 1 RMSE: 1.2279271466210597
HGB Fold 2 RMSE: 1.2298488684480025
HGB Fold 3 RMSE: 1.2173938338304113
HGB Fold 4 RMSE: 1.2260993255870685
HGB Fold 5 RMSE: 1.2453073298043114
HGB Mean RMSE: 1.2293153008581705
HGB OOF RMSE: 1.2293486865124543
LightGBM Fold 1 RMSE: 1.233485237452183
LightGBM Fold 2 RMSE: 1.2333902266646217
LightGBM Fold 3 RMSE: 1.2195575929910218
LightGBM Fold 4 RMSE: 1.2282222267143248
LightGBM Fold 5 RMSE: 1.2492219909564557
LightGBM Mean RMSE: 1.2327754549557215
LightGBM OOF RMSE: 1.2328133212900187


# Deney 22C - v4 Median 3 Seed CatBoost + HGB + LGBM Blend

In [ ]:
from scipy.optimize import minimize

oof_matrix_v4_median_3 = np.column_stack([
    oof_cat_v4_median_3,
    oof_hgb_v4,
    oof_lgbm_v4
])

test_matrix_v4_median_3 = np.column_stack([
    test_cat_v4_median_3,
    test_hgb_v4,
    test_lgbm_v4
])

model_names_v4_median_3 = [
    "CatBoost v4 Median 3 Seed",
    "HGB v4",
    "LightGBM v4"
]

def blend_rmse_v4_median_3(weights):
    preds = oof_matrix_v4_median_3 @ weights
    return rmse(y, preds)

n_models = oof_matrix_v4_median_3.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_v4_median_3 = minimize(
    blend_rmse_v4_median_3,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_v4_median_3 = result_v4_median_3.x
best_oof_preds_v4_median_3 = oof_matrix_v4_median_3 @ best_weights_v4_median_3
best_test_preds_v4_median_3 = test_matrix_v4_median_3 @ best_weights_v4_median_3

print("Optimization success:", result_v4_median_3.success)
print("Best weights:")

for name, weight in zip(model_names_v4_median_3, best_weights_v4_median_3):
    print(f"{name}: {weight:.6f}")

print("Eski en iyi v3 Median 5-seed blend OOF:", rmse(y, best_oof_preds_median_5))
print("v4 Median 3-seed blend OOF:", rmse(y, best_oof_preds_v4_median_3))

Optimization success: True
Best weights:
CatBoost v4 Median 3 Seed: 0.934510
HGB v4: 0.055261
LightGBM v4: 0.010229
Eski en iyi v3 Median 5-seed blend OOF: 1.215180348002664
v4 Median 3-seed blend OOF: 1.215635479269378


# Deney 23 - Target Encoding for LightGBM
### Feature set: X_fe_v3 / X_test_fe_v3

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from category_encoders import TargetEncoder

def get_lgbm_target_encoding_oof(X_fe, y, X_test_fe, cv):
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx].copy()
        X_val_fold = X_fe.iloc[val_idx].copy()
        X_test_fold = X_test_fe.copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # ----------------------------
        # 1. Categorical missing
        # ----------------------------
        for col in cat_cols:
            X_train_fold[col] = X_train_fold[col].fillna("Bilinmiyor").astype(str)
            X_val_fold[col] = X_val_fold[col].fillna("Bilinmiyor").astype(str)
            X_test_fold[col] = X_test_fold[col].fillna("Bilinmiyor").astype(str)

        # ----------------------------
        # 2. Numeric median imputation
        # Sadece train fold'dan öğreniyoruz
        # ----------------------------
        medians = X_train_fold[num_cols].median()

        X_train_fold[num_cols] = X_train_fold[num_cols].fillna(medians)
        X_val_fold[num_cols] = X_val_fold[num_cols].fillna(medians)
        X_test_fold[num_cols] = X_test_fold[num_cols].fillna(medians)

        # ----------------------------
        # 3. Target Encoding
        # Sadece train fold üzerinde fit edilir
        # ----------------------------
        encoder = TargetEncoder(
            cols=cat_cols,
            smoothing=10.0
        )

        X_train_enc = encoder.fit_transform(X_train_fold, y_train_fold)
        X_val_enc = encoder.transform(X_val_fold)
        X_test_enc = encoder.transform(X_test_fold)

        model = LGBMRegressor(
            objective="regression",
            n_estimators=2000,
            learning_rate=0.025,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=25,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.05,
            reg_lambda=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        model.fit(X_train_enc, y_train_fold)

        val_pred = model.predict(X_val_enc)
        test_pred = model.predict(X_test_enc)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"LightGBM TargetEncoding Fold {fold} RMSE:", fold_rmse)

    print("LightGBM TargetEncoding Fold RMSE:", fold_scores)
    print("LightGBM TargetEncoding Mean RMSE:", np.mean(fold_scores))
    print("LightGBM TargetEncoding OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_lgbm_te_v3, test_lgbm_te_v3, scores_lgbm_te_v3 = get_lgbm_target_encoding_oof(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

LightGBM TargetEncoding Fold 1 RMSE: 1.2320133522655294
LightGBM TargetEncoding Fold 2 RMSE: 1.232165919212812
LightGBM TargetEncoding Fold 3 RMSE: 1.2165743791157237
LightGBM TargetEncoding Fold 4 RMSE: 1.2276290508535574
LightGBM TargetEncoding Fold 5 RMSE: 1.2479893559611543
LightGBM TargetEncoding Fold RMSE: [np.float64(1.2320133522655294), np.float64(1.232165919212812), np.float64(1.2165743791157237), np.float64(1.2276290508535574), np.float64(1.2479893559611543)]
LightGBM TargetEncoding Mean RMSE: 1.2312744114817555
LightGBM TargetEncoding OOF RMSE: 1.2313158402004625


# Deney 23B - CatBoost Median 5 Seed + HGB v3 + LightGBM Target Encoding Blend

In [ ]:
from scipy.optimize import minimize

oof_matrix_te = np.column_stack([
    oof_cat_median_5,
    oof_hgb_v3,
    oof_lgbm_te_v3
])

test_matrix_te = np.column_stack([
    test_cat_median_5,
    test_hgb_v3,
    test_lgbm_te_v3
])

model_names_te = [
    "CatBoost v3 Median 5 Seed",
    "HGB v3",
    "LightGBM TargetEncoding v3"
]

def blend_rmse_te(weights):
    preds = oof_matrix_te @ weights
    return rmse(y, preds)

n_models = oof_matrix_te.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_te = minimize(
    blend_rmse_te,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_te = result_te.x
best_oof_preds_te = oof_matrix_te @ best_weights_te
best_test_preds_te = test_matrix_te @ best_weights_te

print("Optimization success:", result_te.success)
print("Best weights:")

for name, weight in zip(model_names_te, best_weights_te):
    print(f"{name}: {weight:.6f}")

print("Eski en iyi Median 5-seed blend OOF:", rmse(y, best_oof_preds_median_5))
print("Target Encoding blend OOF:", rmse(y, best_oof_preds_te))

Optimization success: True
Best weights:
CatBoost v3 Median 5 Seed: 0.915056
HGB v3: 0.037511
LightGBM TargetEncoding v3: 0.047434
Eski en iyi Median 5-seed blend OOF: 1.215180348002664
Target Encoding blend OOF: 1.2151543285184772


# Deney 23C - XGBoost with Out-of-Fold Target Encoding
### Feature set: X_fe_v3 / X_test_fe_v3

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from category_encoders import TargetEncoder

def get_xgb_target_encoding_oof(X_fe, y, X_test_fe, cv):
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx].copy()
        X_val_fold = X_fe.iloc[val_idx].copy()
        X_test_fold = X_test_fe.copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Categorical missing
        for col in cat_cols:
            X_train_fold[col] = X_train_fold[col].fillna("Bilinmiyor").astype(str)
            X_val_fold[col] = X_val_fold[col].fillna("Bilinmiyor").astype(str)
            X_test_fold[col] = X_test_fold[col].fillna("Bilinmiyor").astype(str)

        # Numeric median imputation - sadece train fold üzerinden
        medians = X_train_fold[num_cols].median()

        X_train_fold[num_cols] = X_train_fold[num_cols].fillna(medians)
        X_val_fold[num_cols] = X_val_fold[num_cols].fillna(medians)
        X_test_fold[num_cols] = X_test_fold[num_cols].fillna(medians)

        # Target Encoding - sadece train fold üzerinde fit
        encoder = TargetEncoder(
            cols=cat_cols,
            smoothing=10.0
        )

        X_train_enc = encoder.fit_transform(X_train_fold, y_train_fold)
        X_val_enc = encoder.transform(X_val_fold)
        X_test_enc = encoder.transform(X_test_fold)

        model = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            n_estimators=1500,
            learning_rate=0.03,
            max_depth=5,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            tree_method="hist"
        )

        model.fit(X_train_enc, y_train_fold)

        val_pred = model.predict(X_val_enc)
        test_pred = model.predict(X_test_enc)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"XGBoost TargetEncoding Fold {fold} RMSE:", fold_rmse)

    print("XGBoost TargetEncoding Fold RMSE:", fold_scores)
    print("XGBoost TargetEncoding Mean RMSE:", np.mean(fold_scores))
    print("XGBoost TargetEncoding OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores


oof_xgb_te_v3, test_xgb_te_v3, scores_xgb_te_v3 = get_xgb_target_encoding_oof(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

XGBoost TargetEncoding Fold 1 RMSE: 1.2303676018166907
XGBoost TargetEncoding Fold 2 RMSE: 1.2307845924687602
XGBoost TargetEncoding Fold 3 RMSE: 1.2165291194076377
XGBoost TargetEncoding Fold 4 RMSE: 1.2268646225831656
XGBoost TargetEncoding Fold 5 RMSE: 1.2445348064979067
XGBoost TargetEncoding Fold RMSE: [np.float64(1.2303676018166907), np.float64(1.2307845924687602), np.float64(1.2165291194076377), np.float64(1.2268646225831656), np.float64(1.2445348064979067)]
XGBoost TargetEncoding Mean RMSE: 1.2298161485548322
XGBoost TargetEncoding OOF RMSE: 1.229848928426275


# Deney 23D - CatBoost Median 5 Seed + HGB + LGBM TE + XGB TE Blend

In [ ]:
from scipy.optimize import minimize

oof_matrix_te_xgb = np.column_stack([
    oof_cat_median_5,
    oof_hgb_v3,
    oof_lgbm_te_v3,
    oof_xgb_te_v3
])

test_matrix_te_xgb = np.column_stack([
    test_cat_median_5,
    test_hgb_v3,
    test_lgbm_te_v3,
    test_xgb_te_v3
])

model_names_te_xgb = [
    "CatBoost v3 Median 5 Seed",
    "HGB v3",
    "LightGBM TargetEncoding v3",
    "XGBoost TargetEncoding v3"
]

def blend_rmse_te_xgb(weights):
    preds = oof_matrix_te_xgb @ weights
    return rmse(y, preds)

n_models = oof_matrix_te_xgb.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_te_xgb = minimize(
    blend_rmse_te_xgb,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_te_xgb = result_te_xgb.x
best_oof_preds_te_xgb = oof_matrix_te_xgb @ best_weights_te_xgb
best_test_preds_te_xgb = test_matrix_te_xgb @ best_weights_te_xgb

print("Optimization success:", result_te_xgb.success)
print("Best weights:")

for name, weight in zip(model_names_te_xgb, best_weights_te_xgb):
    print(f"{name}: {weight:.6f}")

print("Eski en iyi TE blend OOF:", rmse(y, best_oof_preds_te))
print("XGB TE dahil blend OOF:", rmse(y, best_oof_preds_te_xgb))

Optimization success: True
Best weights:
CatBoost v3 Median 5 Seed: 0.916096
HGB v3: 0.034082
LightGBM TargetEncoding v3: 0.049822
XGBoost TargetEncoding v3: 0.000000
Eski en iyi TE blend OOF: 1.2151543285184772
XGB TE dahil blend OOF: 1.215154529218457


# Deney 24 - Residual Boosting

In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from category_encoders import TargetEncoder

def get_lgbm_residual_oof(X_fe, y, X_test_fe, cv, base_oof, base_test):
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()

    residual_target = y - base_oof

    oof_residual_preds = np.zeros(len(X_fe))
    test_residual_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx].copy()
        X_val_fold = X_fe.iloc[val_idx].copy()
        X_test_fold = X_test_fe.copy()

        y_train_residual = residual_target.iloc[train_idx]
        y_val_true = y.iloc[val_idx]
        base_val_pred = base_oof[val_idx]

        # Categorical missing
        for col in cat_cols:
            X_train_fold[col] = X_train_fold[col].fillna("Bilinmiyor").astype(str)
            X_val_fold[col] = X_val_fold[col].fillna("Bilinmiyor").astype(str)
            X_test_fold[col] = X_test_fold[col].fillna("Bilinmiyor").astype(str)

        # Numeric median imputation, only train fold
        medians = X_train_fold[num_cols].median()

        X_train_fold[num_cols] = X_train_fold[num_cols].fillna(medians)
        X_val_fold[num_cols] = X_val_fold[num_cols].fillna(medians)
        X_test_fold[num_cols] = X_test_fold[num_cols].fillna(medians)

        # Target encoding residual hedefe göre yapılır
        encoder = TargetEncoder(
            cols=cat_cols,
            smoothing=10.0
        )

        X_train_enc = encoder.fit_transform(X_train_fold, y_train_residual)
        X_val_enc = encoder.transform(X_val_fold)
        X_test_enc = encoder.transform(X_test_fold)

        model = LGBMRegressor(
            objective="regression",
            n_estimators=1500,
            learning_rate=0.02,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=30,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=0.3,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        model.fit(X_train_enc, y_train_residual)

        val_residual_pred = model.predict(X_val_enc)
        test_residual_pred = model.predict(X_test_enc)

        oof_residual_preds[val_idx] = val_residual_pred
        test_residual_preds += test_residual_pred / cv.get_n_splits()

        final_val_pred = base_val_pred + val_residual_pred
        fold_rmse = rmse(y_val_true, final_val_pred)
        fold_scores.append(fold_rmse)

        print(f"LGBM Residual Fold {fold} Final RMSE:", fold_rmse)

    final_oof_preds = base_oof + oof_residual_preds
    final_test_preds = base_test + test_residual_preds

    print("LGBM Residual Fold RMSE:", fold_scores)
    print("LGBM Residual Mean RMSE:", np.mean(fold_scores))
    print("LGBM Residual Final OOF RMSE:", rmse(y, final_oof_preds))

    return final_oof_preds, final_test_preds, oof_residual_preds, test_residual_preds, fold_scores


oof_lgbm_residual_final, test_lgbm_residual_final, oof_lgbm_residual, test_lgbm_residual, scores_lgbm_residual = get_lgbm_residual_oof(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv,
    oof_cat_median_5,
    test_cat_median_5
)

LGBM Residual Fold 1 Final RMSE: 1.232101101786624
LGBM Residual Fold 2 Final RMSE: 1.2318187355529808
LGBM Residual Fold 3 Final RMSE: 1.2171533391009042
LGBM Residual Fold 4 Final RMSE: 1.2260492036337334
LGBM Residual Fold 5 Final RMSE: 1.246521112062133
LGBM Residual Fold RMSE: [np.float64(1.232101101786624), np.float64(1.2318187355529808), np.float64(1.2171533391009042), np.float64(1.2260492036337334), np.float64(1.246521112062133)]
LGBM Residual Mean RMSE: 1.230728698427275
LGBM Residual Final OOF RMSE: 1.2307659652196403


# Deney 24B - Add Residual Model to Existing Best Blend

In [ ]:

from scipy.optimize import minimize

oof_matrix_residual_blend = np.column_stack([
    oof_cat_median_5,
    oof_hgb_v3,
    oof_lgbm_te_v3,
    oof_lgbm_residual_final
])

test_matrix_residual_blend = np.column_stack([
    test_cat_median_5,
    test_hgb_v3,
    test_lgbm_te_v3,
    test_lgbm_residual_final
])

model_names_residual_blend = [
    "CatBoost v3 Median 5 Seed",
    "HGB v3",
    "LightGBM TargetEncoding v3",
    "CatBoost + LGBM Residual Final"
]

def blend_rmse_residual(weights):
    preds = oof_matrix_residual_blend @ weights
    return rmse(y, preds)

n_models = oof_matrix_residual_blend.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_residual_blend = minimize(
    blend_rmse_residual,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_residual_blend = result_residual_blend.x
best_oof_preds_residual_blend = oof_matrix_residual_blend @ best_weights_residual_blend
best_test_preds_residual_blend = test_matrix_residual_blend @ best_weights_residual_blend

print("Optimization success:", result_residual_blend.success)
print("Best weights:")

for name, weight in zip(model_names_residual_blend, best_weights_residual_blend):
    print(f"{name}: {weight:.6f}")

print("Eski en iyi TE blend OOF:", rmse(y, best_oof_preds_te))
print("Residual dahil blend OOF:", rmse(y, best_oof_preds_residual_blend))

Optimization success: True
Best weights:
CatBoost v3 Median 5 Seed: 0.918982
HGB v3: 0.034794
LightGBM TargetEncoding v3: 0.046223
CatBoost + LGBM Residual Final: 0.000000
Eski en iyi TE blend OOF: 1.2151543285184772
Residual dahil blend OOF: 1.2151545065123193


# Deney 25 - Ridge Stacking on OOF Predictions

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

# ------------------------------------------------------------
# 1. Elimizde mevcut olan OOF/test tahminlerini otomatik topla
# ------------------------------------------------------------

stack_oof_list = []
stack_test_list = []
stack_model_names = []

def add_stack_model(name, oof_var_name, test_var_name):
    if oof_var_name in globals() and test_var_name in globals():
        oof_values = globals()[oof_var_name]
        test_values = globals()[test_var_name]

        if len(oof_values) == len(y) and len(test_values) == len(test_ids):
            stack_oof_list.append(oof_values)
            stack_test_list.append(test_values)
            stack_model_names.append(name)
            print(f"Added: {name}")
        else:
            print(f"Skipped {name}: length mismatch")
    else:
        print(f"Skipped {name}: variable not found")


add_stack_model(
    "CatBoost v3 Median 5 Seed",
    "oof_cat_median_5",
    "test_cat_median_5"
)

add_stack_model(
    "HGB v3",
    "oof_hgb_v3",
    "test_hgb_v3"
)

add_stack_model(
    "LightGBM TargetEncoding v3",
    "oof_lgbm_te_v3",
    "test_lgbm_te_v3"
)

add_stack_model(
    "XGBoost TargetEncoding v3",
    "oof_xgb_te_v3",
    "test_xgb_te_v3"
)

add_stack_model(
    "LightGBM v3",
    "oof_lgbm_v3",
    "test_lgbm_v3"
)

add_stack_model(
    "CatBoost + LGBM Residual Final",
    "oof_lgbm_residual_final",
    "test_lgbm_residual_final"
)

X_stack_oof = np.column_stack(stack_oof_list)
X_stack_test = np.column_stack(stack_test_list)

print("Stacking models:", stack_model_names)
print("X_stack_oof:", X_stack_oof.shape)
print("X_stack_test:", X_stack_test.shape)

# ------------------------------------------------------------
# 2. Meta model için OOF değerlendirme
# Burada Ridge'i de OOF mantığıyla test ediyoruz.
# ------------------------------------------------------------

meta_cv = KFold(n_splits=5, shuffle=True, random_state=42)

alphas = np.logspace(-4, 4, 30)

ridge_oof_preds = np.zeros(len(y))
ridge_fold_scores = []
ridge_fold_alphas = []
ridge_fold_coefs = []

for fold, (train_idx, val_idx) in enumerate(meta_cv.split(X_stack_oof), 1):
    X_train_meta = X_stack_oof[train_idx]
    X_val_meta = X_stack_oof[val_idx]

    y_train_meta = y.iloc[train_idx]
    y_val_meta = y.iloc[val_idx]

    ridge_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("ridge", RidgeCV(alphas=alphas))
    ])

    ridge_model.fit(X_train_meta, y_train_meta)

    val_pred = ridge_model.predict(X_val_meta)
    ridge_oof_preds[val_idx] = val_pred

    fold_rmse = rmse(y_val_meta, val_pred)
    ridge_fold_scores.append(fold_rmse)

    best_alpha = ridge_model.named_steps["ridge"].alpha_
    coefs = ridge_model.named_steps["ridge"].coef_

    ridge_fold_alphas.append(best_alpha)
    ridge_fold_coefs.append(coefs)

    print(f"Ridge Stacking Fold {fold} RMSE:", fold_rmse)
    print(f"Fold {fold} alpha:", best_alpha)

print("Ridge Stacking Fold RMSE:", ridge_fold_scores)
print("Ridge Stacking Mean RMSE:", np.mean(ridge_fold_scores))
print("Ridge Stacking OOF RMSE:", rmse(y, ridge_oof_preds))



if "best_oof_preds_te" in globals():
    print("Current best TE blend OOF:", rmse(y, best_oof_preds_te))

if "best_oof_preds_median_5" in globals():
    print("Median 5-seed blend OOF:", rmse(y, best_oof_preds_median_5))



coef_df = pd.DataFrame(
    ridge_fold_coefs,
    columns=stack_model_names
)

print("\nAverage Ridge coefficients:")
display(coef_df.mean().sort_values(ascending=False))

print("\nFold alphas:")
print(ridge_fold_alphas)


final_ridge_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=alphas))
])

final_ridge_model.fit(X_stack_oof, y)

best_oof_preds_ridge_stack = ridge_oof_preds
best_test_preds_ridge_stack = final_ridge_model.predict(X_stack_test)

best_test_preds_ridge_stack = np.clip(best_test_preds_ridge_stack, 0, 10)

print("\nFinal Ridge alpha:", final_ridge_model.named_steps["ridge"].alpha_)
print("Final Ridge coefficients:")

final_coef_df = pd.DataFrame({
    "model": stack_model_names,
    "coef": final_ridge_model.named_steps["ridge"].coef_
}).sort_values("coef", ascending=False)

display(final_coef_df)

print("Ridge stacked test prediction summary:")
print(pd.Series(best_test_preds_ridge_stack).describe())

Added: CatBoost v3 Median 5 Seed
Added: HGB v3
Added: LightGBM TargetEncoding v3
Added: XGBoost TargetEncoding v3
Added: LightGBM v3
Added: CatBoost + LGBM Residual Final
Stacking models: ['CatBoost v3 Median 5 Seed', 'HGB v3', 'LightGBM TargetEncoding v3', 'XGBoost TargetEncoding v3', 'LightGBM v3', 'CatBoost + LGBM Residual Final']
X_stack_oof: (56000, 6)
X_stack_test: (24000, 6)
Ridge Stacking Fold 1 RMSE: 1.2192282930908693
Fold 1 alpha: 0.7278953843983146
Ridge Stacking Fold 2 RMSE: 1.2137686422510456
Fold 2 alpha: 0.7278953843983146
Ridge Stacking Fold 3 RMSE: 1.198562222003559
Fold 3 alpha: 0.7278953843983146
Ridge Stacking Fold 4 RMSE: 1.2064136993627548
Fold 4 alpha: 0.7278953843983146
Ridge Stacking Fold 5 RMSE: 1.22889688671241
Fold 5 alpha: 0.7278953843983146
Ridge Stacking Fold RMSE: [np.float64(1.2192282930908693), np.float64(1.2137686422510456), np.float64(1.198562222003559), np.float64(1.2064136993627548), np.float64(1.22889688671241)]
Ridge Stacking Mean RMSE: 1.213373

CatBoost v3 Median 5 Seed         2.389292
LightGBM v3                       0.283433
LightGBM TargetEncoding v3        0.187199
XGBoost TargetEncoding v3         0.091384
HGB v3                           -0.112863
CatBoost + LGBM Residual Final   -0.966725
dtype: float64


Fold alphas:
[0.7278953843983146, 0.7278953843983146, 0.7278953843983146, 0.7278953843983146, 0.7278953843983146]

Final Ridge alpha: 0.7278953843983146
Final Ridge coefficients:


,model,coef
0,CatBoost v3 Median 5 Seed,2.392633
4,LightGBM v3,0.283836
2,LightGBM TargetEncoding v3,0.187594
3,XGBoost TargetEncoding v3,0.090863
1,HGB v3,-0.114132
5,CatBoost + LGBM Residual Final,-0.969080


Ridge stacked test prediction summary:
count    24000.000000
mean         5.939449
std          1.870360
min          0.000000
25%          4.635793
50%          6.051575
75%          7.333041
max         10.000000
dtype: float64


In [ ]:
submission_exp25_ridge = pd.DataFrame({
    "id": test_ids,
    TARGET: best_test_preds_ridge_stack
})

submission_exp25_ridge[TARGET] = submission_exp25_ridge[TARGET].clip(0, 10)

print(submission_exp25_ridge.shape)
print(submission_exp25_ridge.head())
print(submission_exp25_ridge[TARGET].describe())

submission_exp25_ridge.to_csv("submission_exp25_ridge_stacking.csv", index=False)

(24000, 2)
   id  bilissel_performans_skoru
0   1                   6.035980
1   2                   6.723272
2   3                   2.960330
3   4                   7.136152
4   5                   3.638989
count    24000.000000
mean         5.939449
std          1.870360
min          0.000000
25%          4.635793
50%          6.051575
75%          7.333041
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


In [ ]:
import pandas as pd
import numpy as np

TARGET = "bilissel_performans_skoru"

In [ ]:
import pandas as pd
import numpy as np

TARGET = "bilissel_performans_skoru"

exp20e = pd.read_csv("submission_exp20e_v3_5seed_blend.csv")

print("exp20e public best prediction:")
print(exp20e[TARGET].describe())

print("\nexp23b te blend prediction:")

submission_exp23b_te_blend = pd.DataFrame({
    "id": exp20e["id"],
    TARGET: best_test_preds_te
})

submission_exp23b_te_blend[TARGET] = submission_exp23b_te_blend[TARGET].clip(0, 10)

print(submission_exp23b_te_blend[TARGET].describe())

diff = submission_exp23b_te_blend[TARGET].values - exp20e[TARGET].values

print("\nDifference exp23b - exp20e:")
print(pd.Series(diff).describe())

print("\nCorrelation:")
print(np.corrcoef(exp20e[TARGET].values, submission_exp23b_te_blend[TARGET].values)[0, 1])

exp20e public best prediction:
count    24000.000000
mean         5.936977
std          1.862729
min          0.000000
25%          4.641429
50%          6.044348
75%          7.327849
max         10.000000
Name: bilissel_performans_skoru, dtype: float64

exp23b te blend prediction:
count    24000.000000
mean         5.935936
std          1.865624
min          0.000000
25%          4.641019
50%          6.046583
75%          7.325603
max         10.000000
Name: bilissel_performans_skoru, dtype: float64

Difference exp23b - exp20e:
count    24000.000000
mean        -0.001042
std          0.034862
min         -0.416018
25%         -0.019370
50%         -0.002413
75%          0.015080
max          0.378444
dtype: float64

Correlation:
0.9998263421694981


# Deney 26 - Target Mechanism Based Feature Engineering

In [ ]:
# ============================================================
# CatBoost OOF with Numeric Median Imputation
# Kullanım: v3 / v5 gibi feature setleriyle çalışır
# ============================================================

from catboost import CatBoostRegressor
import numpy as np

def get_catboost_oof_with_seed_median(X_fe, y, X_test_fe, cv, seed):
    X_cb = X_fe.copy()
    X_test_cb = X_test_fe.copy()

    cat_cols = X_cb.select_dtypes(include="object").columns.tolist()
    num_cols = X_cb.select_dtypes(include=np.number).columns.tolist()


    for col in cat_cols:
        X_cb[col] = X_cb[col].fillna("Bilinmiyor").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("Bilinmiyor").astype(str)

    cat_features_idx = [X_cb.columns.get_loc(col) for col in cat_cols]

    oof_preds = np.zeros(len(X_cb))
    test_preds = np.zeros(len(X_test_cb))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_cb), 1):
        X_train_fold = X_cb.iloc[train_idx].copy()
        X_val_fold = X_cb.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        X_test_fold = X_test_cb.copy()

        medians = X_train_fold[num_cols].median()

        X_train_fold[num_cols] = X_train_fold[num_cols].fillna(medians)
        X_val_fold[num_cols] = X_val_fold[num_cols].fillna(medians)
        X_test_fold[num_cols] = X_test_fold[num_cols].fillna(medians)

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=1500,
            learning_rate=0.035,
            depth=6,
            l2_leaf_reg=5,
            random_seed=seed,
            verbose=200,
            early_stopping_rounds=100
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features_idx,
            eval_set=(X_val_fold, y_val_fold),
            use_best_model=True
        )

        val_pred = model.predict(X_val_fold)
        test_pred = model.predict(X_test_fold)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} - Fold {fold} RMSE:", fold_rmse)

    print(f"Seed {seed} Mean RMSE:", np.mean(fold_scores))
    print(f"Seed {seed} OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test_x.csv")

TARGET = "bilissel_performans_skoru"

X = train.drop(columns=[TARGET])
X_test = test.copy()
y = train[TARGET]

test_ids = test["id"]

print("Train:", train.shape)
print("Test:", test.shape)
print("X:", X.shape)
print("X_test:", X_test.shape)

Train: (56000, 24)
Test: (24000, 23)
X: (56000, 23)
X_test: (24000, 23)


In [ ]:
# ============================================================
# Feature v3
# v1 feature set + missing indicators + categorical interactions
# ============================================================

def prepare_features_v3(X, X_test):
    X_fe = X.copy()
    X_test_fe = X_test.copy()

    # --------------------------------------------------------
    # 1. Missing indicator'ları ekle
    # --------------------------------------------------------
    all_cols = X_fe.columns.tolist()

    for col in all_cols:
        if X_fe[col].isnull().any() or X_test_fe[col].isnull().any():
            X_fe[f"{col}_missing"] = X_fe[col].isnull().astype(int)
            X_test_fe[f"{col}_missing"] = X_test_fe[col].isnull().astype(int)

    # --------------------------------------------------------
    # 2. Log transform - v1'de işe yarayan dönüşüm
    # --------------------------------------------------------
    log_cols = [
        "uyku_oncesi_kafein_mg",
        "uyku_oncesi_ekran_suresi_dk"
    ]

    for col in log_cols:
        if col in X_fe.columns:
            X_fe[col] = np.log1p(X_fe[col])
            X_test_fe[col] = np.log1p(X_test_fe[col])

    def add_features(df):
        df = df.copy()

        # ----------------------------------------------------
        # v1'de işe yarayan feature'lar
        # ----------------------------------------------------
        if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]

        if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
            df["uyku_kalitesi_orani"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

        if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
            df["uyku_bozulma_skoru"] = df["gecelik_uyanma_sayisi"] + df["uykuya_dalma_suresi_dk"]

        if "uyku_oncesi_ekran_suresi_dk" in df.columns and "uyku_oncesi_kafein_mg" in df.columns:
            df["dijital_kafein_yuku"] = (
                df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]
            )

        if "gunluk_adim_sayisi" in df.columns and "stres_skoru" in df.columns:
            df["aktivite_stres_orani"] = df["gunluk_adim_sayisi"] / (df["stres_skoru"] + 1)

        # ----------------------------------------------------
        # Kategorik interaction feature'lar
        # ----------------------------------------------------
        def safe_str_col(c):
            return df[c].fillna("Bilinmiyor").astype(str)

        if "meslek" in df.columns and "kronotip" in df.columns:
            df["meslek_kronotip"] = safe_str_col("meslek") + "_" + safe_str_col("kronotip")

        if "gun_tipi" in df.columns and "kronotip" in df.columns:
            df["gun_tipi_kronotip"] = safe_str_col("gun_tipi") + "_" + safe_str_col("kronotip")

        if "ruh_sagligi_durumu" in df.columns and "meslek" in df.columns:
            df["ruh_sagligi_meslek"] = safe_str_col("ruh_sagligi_durumu") + "_" + safe_str_col("meslek")

        if "cinsiyet" in df.columns and "kronotip" in df.columns:
            df["cinsiyet_kronotip"] = safe_str_col("cinsiyet") + "_" + safe_str_col("kronotip")

        return df

    X_fe = add_features(X_fe)
    X_test_fe = add_features(X_test_fe)

    return X_fe, X_test_fe

In [ ]:
X_fe_v3, X_test_fe_v3 = prepare_features_v3(X, X_test)

print("X_fe_v3:", X_fe_v3.shape)
print("X_test_fe_v3:", X_test_fe_v3.shape)

X_fe_v3: (56000, 38)
X_test_fe_v3: (24000, 38)


In [ ]:
# ============================================================
# Deney 26 - Feature v5
# Target mechanism based controlled interactions
# Base: feature v3
# ============================================================

def prepare_features_v5_target_mechanism(X, X_test):

    X_fe, X_test_fe = prepare_features_v3(X, X_test)

    def add_target_mechanism_features(df):
        df = df.copy()



        if "rem_yuzdesi" in df.columns and "derin_uyku_yuzdesi" in df.columns:
            df["toplam_kaliteli_uyku"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]

        if "gecelik_uyanma_sayisi" in df.columns and "uykuya_dalma_suresi_dk" in df.columns:
            df["uyku_kesinti_yuku"] = (
                df["gecelik_uyanma_sayisi"] * 2
                + df["uykuya_dalma_suresi_dk"] / 5
            )

        if "stres_skoru" in df.columns and "gunluk_calisma_saati" in df.columns:
            df["zihinsel_yuk"] = (
                df["stres_skoru"] * 2
                + df["gunluk_calisma_saati"]
            )

        if (
            "stres_skoru" in df.columns
            and "gunluk_calisma_saati" in df.columns
            and "uyku_kesinti_yuku" in df.columns
        ):
            df["toplam_risk_yuku"] = (
                df["stres_skoru"] * 2
                + df["gunluk_calisma_saati"]
                + df["uyku_kesinti_yuku"]
            )

        if "toplam_kaliteli_uyku" in df.columns and "toplam_risk_yuku" in df.columns:
            df["net_bilissel_denge"] = (
                df["toplam_kaliteli_uyku"] - df["toplam_risk_yuku"]
            )

        if "toplam_kaliteli_uyku" in df.columns and "stres_skoru" in df.columns:
            df["uyku_stres_dengesi"] = (
                df["toplam_kaliteli_uyku"] / (df["stres_skoru"] + 1)
            )

        if "gunluk_adim_sayisi" in df.columns and "zihinsel_yuk" in df.columns:
            df["aktivite_zihinsel_yuk_dengesi"] = (
                df["gunluk_adim_sayisi"] / (df["zihinsel_yuk"] + 1)
            )



        def safe_str_col(c):
            return df[c].fillna("Bilinmiyor").astype(str)

        if "ruh_sagligi_durumu" in df.columns and "gun_tipi" in df.columns:
            df["ruh_sagligi_gun_tipi"] = (
                safe_str_col("ruh_sagligi_durumu")
                + "_"
                + safe_str_col("gun_tipi")
            )

        if "meslek" in df.columns and "gun_tipi" in df.columns:
            df["meslek_gun_tipi"] = (
                safe_str_col("meslek")
                + "_"
                + safe_str_col("gun_tipi")
            )

        if "ruh_sagligi_durumu" in df.columns and "kronotip" in df.columns:
            df["ruh_sagligi_kronotip"] = (
                safe_str_col("ruh_sagligi_durumu")
                + "_"
                + safe_str_col("kronotip")
            )


        if "gun_tipi" in df.columns:
            is_weekend = (safe_str_col("gun_tipi") == "Hafta sonu").astype(int)

            if "toplam_kaliteli_uyku" in df.columns:
                df["hafta_sonu_uyku_kalitesi"] = df["toplam_kaliteli_uyku"] * is_weekend

            if "stres_skoru" in df.columns:
                df["hafta_sonu_stres"] = df["stres_skoru"] * is_weekend

            if "gunluk_calisma_saati" in df.columns:
                df["hafta_sonu_calisma"] = df["gunluk_calisma_saati"] * is_weekend

        if "ruh_sagligi_durumu" in df.columns:
            mental = safe_str_col("ruh_sagligi_durumu")

            is_saglikli = (mental == "Saglikli").astype(int)
            is_depresyon = mental.str.contains("Depresyon", case=False, na=False).astype(int)
            is_anksiyete = mental.str.contains("Anksiyete", case=False, na=False).astype(int)

            if "stres_skoru" in df.columns:
                df["stres_saglikli"] = df["stres_skoru"] * is_saglikli
                df["stres_depresyon"] = df["stres_skoru"] * is_depresyon
                df["stres_anksiyete"] = df["stres_skoru"] * is_anksiyete

            if "uyku_kesinti_yuku" in df.columns:
                df["uyku_kesinti_depresyon"] = df["uyku_kesinti_yuku"] * is_depresyon
                df["uyku_kesinti_anksiyete"] = df["uyku_kesinti_yuku"] * is_anksiyete

        return df

    X_fe = add_target_mechanism_features(X_fe)
    X_test_fe = add_target_mechanism_features(X_test_fe)

    return X_fe, X_test_fe


X_fe_v5, X_test_fe_v5 = prepare_features_v5_target_mechanism(X, X_test)

print("X_fe_v5:", X_fe_v5.shape)
print("X_test_fe_v5:", X_test_fe_v5.shape)
print("New columns vs original:", X_fe_v5.shape[1] - X.shape[1])
print("New columns vs v3:", X_fe_v5.shape[1] - X_fe_v3.shape[1])

X_fe_v5: (56000, 56)
X_test_fe_v5: (24000, 56)
New columns vs original: 33
New columns vs v3: 18


In [ ]:
# ============================================================
# Deney 26A - CatBoost v5 + Median Imputation - 3 Seed
# ============================================================

seeds_v5_median_3 = [42, 2024, 3407]

cat_oof_list_v5_median_3 = []
cat_test_list_v5_median_3 = []
cat_seed_scores_v5_median_3 = {}

for seed in seeds_v5_median_3:
    print("=" * 70)
    print(f"Running CatBoost v5 median seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed_median(
        X_fe_v5,
        y,
        X_test_fe_v5,
        cv,
        seed
    )

    cat_oof_list_v5_median_3.append(oof_seed)
    cat_test_list_v5_median_3.append(test_seed)

    cat_seed_scores_v5_median_3[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_v5_median_3 = np.mean(cat_oof_list_v5_median_3, axis=0)
test_cat_v5_median_3 = np.mean(cat_test_list_v5_median_3, axis=0)

print("=" * 70)
print("CatBoost v5 Median 3 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_v5_median_3.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("Eski v3 Median 3-seed CatBoost OOF:", rmse(y, oof_cat_median_3))
print("v5 Median 3-seed CatBoost OOF:", rmse(y, oof_cat_v5_median_3))

In [ ]:
print("v5 Median 3-seed CatBoost OOF:", rmse(y, oof_cat_v5_median_3))

for seed, result in cat_seed_scores_v5_median_3.items():
    print(seed, result["oof_rmse"])


v5 Median 3-seed CatBoost OOF: 1.215253009557772
42 1.2161361543543805
2024 1.2158896104864432
3407 1.216337008606478


# Deney 26B - CatBoost v5 + Median Imputation - 5 Seed

In [31]:
seeds_v5_median_5 = [42, 2024, 3407, 777, 999]

cat_oof_list_v5_median_5 = []
cat_test_list_v5_median_5 = []
cat_seed_scores_v5_median_5 = {}

for seed in seeds_v5_median_5:
    print("=" * 70)
    print(f"Running CatBoost v5 median seed: {seed}")
    print("=" * 70)

    oof_seed, test_seed, scores_seed = get_catboost_oof_with_seed_median(
        X_fe_v5,
        y,
        X_test_fe_v5,
        cv,
        seed
    )

    cat_oof_list_v5_median_5.append(oof_seed)
    cat_test_list_v5_median_5.append(test_seed)

    cat_seed_scores_v5_median_5[seed] = {
        "fold_scores": scores_seed,
        "mean_rmse": np.mean(scores_seed),
        "oof_rmse": rmse(y, oof_seed)
    }

oof_cat_v5_median_5 = np.mean(cat_oof_list_v5_median_5, axis=0)
test_cat_v5_median_5 = np.mean(cat_test_list_v5_median_5, axis=0)

print("=" * 70)
print("CatBoost v5 Median 5 Seed Scores")
print("=" * 70)

for seed, result in cat_seed_scores_v5_median_5.items():
    print(f"Seed {seed}")
    print("Mean RMSE:", result["mean_rmse"])
    print("OOF RMSE:", result["oof_rmse"])
    print()

print("v5 Median 3-seed CatBoost OOF: 1.215253009557772")
print("v3 Median 5-seed CatBoost OOF: 1.2152369352820866")
print("v5 Median 5-seed CatBoost OOF:", rmse(y, oof_cat_v5_median_5))

Running CatBoost v5 median seed: 42
0:	learn: 2.1860157	test: 2.1969404	best: 2.1969404 (0)	total: 12.1ms	remaining: 18.1s
200:	learn: 1.2212698	test: 1.2398277	best: 1.2398277 (200)	total: 2.64s	remaining: 17.1s
400:	learn: 1.1987452	test: 1.2264998	best: 1.2264998 (400)	total: 6.09s	remaining: 16.7s
600:	learn: 1.1864098	test: 1.2236785	best: 1.2236785 (600)	total: 8.91s	remaining: 13.3s
800:	learn: 1.1757868	test: 1.2223439	best: 1.2223439 (800)	total: 11.8s	remaining: 10.3s
1000:	learn: 1.1658999	test: 1.2219821	best: 1.2219821 (1000)	total: 14.6s	remaining: 7.29s
1200:	learn: 1.1569284	test: 1.2215944	best: 1.2215544 (1194)	total: 17.6s	remaining: 4.38s
1400:	learn: 1.1487995	test: 1.2216265	best: 1.2214573 (1304)	total: 20.5s	remaining: 1.45s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.221457297
bestIteration = 1304

Shrink model to first 1305 iterations.
Seed 42 - Fold 1 RMSE: 1.2214572977120415
0:	learn: 2.1893572	test: 2.1805752	best: 2.1805752 (0)	tot

# Deney 26C - v5 CatBoost 5 Seed + HGB v3 + LightGBM TE v3 Blend

In [33]:
for var in [
    "oof_cat_v5_median_5",
    "test_cat_v5_median_5",
    "oof_hgb_v3",
    "test_hgb_v3",
    "oof_lgbm_te_v3",
    "test_lgbm_te_v3"
]:
    print(var, var in globals())

oof_cat_v5_median_5 True
test_cat_v5_median_5 True
oof_hgb_v3 False
test_hgb_v3 False
oof_lgbm_te_v3 False
test_lgbm_te_v3 False


In [34]:
for var in [
    "X_fe_v3",
    "X_test_fe_v3",
    "get_hgb_oof_dense",
    "get_lgbm_target_encoding_oof"
]:
    print(var, var in globals())

X_fe_v3 True
X_test_fe_v3 True
get_hgb_oof_dense False
get_lgbm_target_encoding_oof False


In [ ]:
# ============================================================
# HGB OOF - Dense OneHotEncoder
# ============================================================

import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

def get_hgb_oof_dense(X_fe, y, X_test_fe, cv):
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Bilinmiyor")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ],
        sparse_threshold=0
    )

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx]
        X_val_fold = X_fe.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model = HistGradientBoostingRegressor(
            max_iter=700,
            learning_rate=0.04,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.1,
            random_state=42
        )

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_pred = pipeline.predict(X_val_fold)
        test_pred = pipeline.predict(X_test_fe)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"HGB Fold {fold} RMSE:", fold_rmse)

    print("HGB Mean RMSE:", np.mean(fold_scores))
    print("HGB OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores

In [ ]:
# ============================================================
# LightGBM with Out-of-Fold Target Encoding
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from category_encoders import TargetEncoder

def get_lgbm_target_encoding_oof(X_fe, y, X_test_fe, cv):
    cat_cols = X_fe.select_dtypes(include="object").columns.tolist()
    num_cols = X_fe.select_dtypes(include=np.number).columns.tolist()

    oof_preds = np.zeros(len(X_fe))
    test_preds = np.zeros(len(X_test_fe))
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_fe), 1):
        X_train_fold = X_fe.iloc[train_idx].copy()
        X_val_fold = X_fe.iloc[val_idx].copy()
        X_test_fold = X_test_fe.copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]


        for col in cat_cols:
            X_train_fold[col] = X_train_fold[col].fillna("Bilinmiyor").astype(str)
            X_val_fold[col] = X_val_fold[col].fillna("Bilinmiyor").astype(str)
            X_test_fold[col] = X_test_fold[col].fillna("Bilinmiyor").astype(str)

        medians = X_train_fold[num_cols].median()

        X_train_fold[num_cols] = X_train_fold[num_cols].fillna(medians)
        X_val_fold[num_cols] = X_val_fold[num_cols].fillna(medians)
        X_test_fold[num_cols] = X_test_fold[num_cols].fillna(medians)

        encoder = TargetEncoder(
            cols=cat_cols,
            smoothing=10.0
        )

        X_train_enc = encoder.fit_transform(X_train_fold, y_train_fold)
        X_val_enc = encoder.transform(X_val_fold)
        X_test_enc = encoder.transform(X_test_fold)

        model = LGBMRegressor(
            objective="regression",
            n_estimators=2000,
            learning_rate=0.025,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=25,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_alpha=0.05,
            reg_lambda=0.1,
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

        model.fit(X_train_enc, y_train_fold)

        val_pred = model.predict(X_val_enc)
        test_pred = model.predict(X_test_enc)

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / cv.get_n_splits()

        fold_rmse = rmse(y_val_fold, val_pred)
        fold_scores.append(fold_rmse)

        print(f"LightGBM TargetEncoding Fold {fold} RMSE:", fold_rmse)

    print("LightGBM TargetEncoding Mean RMSE:", np.mean(fold_scores))
    print("LightGBM TargetEncoding OOF RMSE:", rmse(y, oof_preds))

    return oof_preds, test_preds, fold_scores

In [37]:
oof_hgb_v3, test_hgb_v3, scores_hgb_v3 = get_hgb_oof_dense(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

oof_lgbm_te_v3, test_lgbm_te_v3, scores_lgbm_te_v3 = get_lgbm_target_encoding_oof(
    X_fe_v3,
    y,
    X_test_fe_v3,
    cv
)

HGB Fold 1 RMSE: 1.2307257608746391
HGB Fold 2 RMSE: 1.2279666280094808
HGB Fold 3 RMSE: 1.2180883161193514
HGB Fold 4 RMSE: 1.2249099451212055
HGB Fold 5 RMSE: 1.2450940604787741
HGB Mean RMSE: 1.22935694212069
HGB OOF RMSE: 1.2293893343402866
LightGBM TargetEncoding Fold 1 RMSE: 1.2321067384815605
LightGBM TargetEncoding Fold 2 RMSE: 1.2303617074700182
LightGBM TargetEncoding Fold 3 RMSE: 1.216124730155015
LightGBM TargetEncoding Fold 4 RMSE: 1.2273107472783782
LightGBM TargetEncoding Fold 5 RMSE: 1.2479365954034762
LightGBM TargetEncoding Mean RMSE: 1.2307681037576894
LightGBM TargetEncoding OOF RMSE: 1.2308106046055216


In [38]:
from scipy.optimize import minimize
import numpy as np

oof_matrix_v5_hybrid = np.column_stack([
    oof_cat_v5_median_5,
    oof_hgb_v3,
    oof_lgbm_te_v3
])

test_matrix_v5_hybrid = np.column_stack([
    test_cat_v5_median_5,
    test_hgb_v3,
    test_lgbm_te_v3
])

model_names_v5_hybrid = [
    "CatBoost v5 Median 5 Seed",
    "HGB v3",
    "LightGBM TargetEncoding v3"
]

def blend_rmse_v5_hybrid(weights):
    preds = oof_matrix_v5_hybrid @ weights
    return rmse(y, preds)

n_models = oof_matrix_v5_hybrid.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_v5_hybrid = minimize(
    blend_rmse_v5_hybrid,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_v5_hybrid = result_v5_hybrid.x
best_oof_preds_v5_hybrid = oof_matrix_v5_hybrid @ best_weights_v5_hybrid
best_test_preds_v5_hybrid = test_matrix_v5_hybrid @ best_weights_v5_hybrid

print("Optimization success:", result_v5_hybrid.success)
print("Best weights:")

for name, weight in zip(model_names_v5_hybrid, best_weights_v5_hybrid):
    print(f"{name}: {weight:.6f}")

print("Eski en iyi TE blend OOF: 1.2151543285184772")
print("v5 hybrid blend OOF:", rmse(y, best_oof_preds_v5_hybrid))

Optimization success: True
Best weights:
CatBoost v5 Median 5 Seed: 0.844683
HGB v3: 0.059198
LightGBM TargetEncoding v3: 0.096119
Eski en iyi TE blend OOF: 1.2151543285184772
v5 hybrid blend OOF: 1.2148729252291883


In [39]:
import pandas as pd
import numpy as np

TARGET = "bilissel_performans_skoru"

exp20e = pd.read_csv("submission_exp20e_v3_5seed_blend.csv")

submission_exp26c_v5_hybrid = pd.DataFrame({
    "id": exp20e["id"],
    TARGET: best_test_preds_v5_hybrid
})

submission_exp26c_v5_hybrid[TARGET] = submission_exp26c_v5_hybrid[TARGET].clip(0, 10)

print("exp20E public best:")
print(exp20e[TARGET].describe())

print("\nexp26C v5 hybrid:")
print(submission_exp26c_v5_hybrid[TARGET].describe())

diff = submission_exp26c_v5_hybrid[TARGET].values - exp20e[TARGET].values

print("\nDifference exp26C - exp20E:")
print(pd.Series(diff).describe())

print("\nCorrelation:")
print(np.corrcoef(exp20e[TARGET].values, submission_exp26c_v5_hybrid[TARGET].values)[0, 1])

exp20E public best:
count    24000.000000
mean         5.936977
std          1.862729
min          0.000000
25%          4.641429
50%          6.044348
75%          7.327849
max         10.000000
Name: bilissel_performans_skoru, dtype: float64

exp26C v5 hybrid:
count    24000.000000
mean         5.942902
std          1.862127
min          0.036221
25%          4.639724
50%          6.041753
75%          7.340950
max         10.000000
Name: bilissel_performans_skoru, dtype: float64

Difference exp26C - exp20E:
count    24000.000000
mean         0.005925
std          0.087670
min         -0.515707
25%         -0.045300
50%          0.003504
75%          0.055709
max          0.573976
dtype: float64

Correlation:
0.9988921212494891


In [40]:
submission_exp26c_v5_hybrid.to_csv("submission_exp26c_v5_hybrid.csv", index=False)

# Deney 27 - Public Best Çizgisini Baştan Kurma

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor, Pool


# ============================================================
# EXP 27 - Clean CatBoost Native Pipeline
# ============================================================

EXP_NAME = "exp27_cat_native_clean"
SEEDS = [42, 2024, 3407, 777, 999]
N_SPLITS = 5

TARGET_COL = "Degerlendirme Puani"


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)



cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

print("Categorical features:", cat_features)
print("Number of categorical features:", len(cat_features))

print("Remaining categorical NaN in train:", X[cat_features].isna().sum().sum())
print("Remaining categorical NaN in test:", X_test[cat_features].isna().sum().sum())


cat_oof_list_exp27 = []
cat_test_list_exp27 = []
cat_seed_scores_exp27 = {}


for seed in SEEDS:
    print("=" * 70)
    print(f"Starting seed: {seed}")
    print("=" * 70)

    kf = KFold(
        n_splits=N_SPLITS,cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical features:", cat_features)
print("Number of categorical features:", len(cat_features))
    test_pred = np.zeros(len(X_test))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), start=1):
        print(f"\nSeed {seed} | Fold {fold}")

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        train_pool = Pool(
            X_train,
            y_train,
            cat_features=cat_features
        )

        valid_pool = Pool(
            X_valid,
            y_valid,
            cat_features=cat_features
        )

        test_pool = Pool(
            X_test,
            cat_features=cat_features
        )

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=5000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=3,
            random_seed=seed,
            verbose=500,
            early_stopping_rounds=300,
            allow_writing_files=False
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True
        )

        valid_pred = model.predict(valid_pool)
        fold_test_pred = model.predict(test_pool)

        oof_pred[valid_idx] = valid_pred
        test_pred += fold_test_pred / N_SPLITS

        fold_rmse = rmse(y_valid, valid_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} | Fold {fold} RMSE: {fold_rmse:.5f}")

    seed_rmse = rmse(y, oof_pred)

    cat_oof_list_exp27.append(oof_pred)
    cat_test_list_exp27.append(test_pred)
    cat_seed_scores_exp27[seed] = {
        "fold_scores": fold_scores,
        "seed_rmse": seed_rmse
    }

    print(f"\nSeed {seed} OOF RMSE: {seed_rmse:.5f}")


# Güvenlik kontrolü
assert len(cat_oof_list_exp27) > 0, "cat_oof_list_exp27 boş! Model eğitimi çalışmamış."
assert len(cat_test_list_exp27) > 0, "cat_test_list_exp27 boş! Test tahminleri oluşmamış."


# 5 seed ortalaması
oof_cat_exp27_5seed = np.mean(cat_oof_list_exp27, axis=0)
test_cat_exp27_5seed = np.mean(cat_test_list_exp27, axis=0)

final_rmse = rmse(y, oof_cat_exp27_5seed)

print("\n" + "=" * 70)
print(f"{EXP_NAME} Final 5-seed OOF RMSE: {final_rmse:.5f}")
print("=" * 70)


# Kaydet
np.save(f"oof_{EXP_NAME}.npy", oof_cat_exp27_5seed)
np.save(f"test_{EXP_NAME}.npy", test_cat_exp27_5seed)

print(f"Saved: oof_{EXP_NAME}.npy")
print(f"Saved: test_{EXP_NAME}.npy")

X shape: (56000, 23)
y shape: (56000,)
X_test shape: (24000, 23)
Categorical features: ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
Number of categorical features: 7
Remaining categorical NaN in train: 0
Remaining categorical NaN in test: 0
Starting seed: 42

Seed 42 | Fold 1
0:	learn: 2.1962065	test: 2.2069443	best: 2.2069443 (0)	total: 53.9ms	remaining: 4m 29s
500:	learn: 1.1958354	test: 1.2241326	best: 1.2241326 (500)	total: 4.65s	remaining: 41.8s
1000:	learn: 1.1711294	test: 1.2209564	best: 1.2209564 (1000)	total: 8.39s	remaining: 33.5s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 1.220786278
bestIteration = 1079

Shrink model to first 1080 iterations.
Seed 42 | Fold 1 RMSE: 1.22079

Seed 42 | Fold 2
0:	learn: 2.1998516	test: 2.1910527	best: 2.1910527 (0)	total: 7.03ms	remaining: 35.2s
500:	learn: 1.1975040	test: 1.2199261	best: 1.2199241 (498)	total: 3.69s	remaining: 33.2s
1000:	learn: 1.1703787	test: 1.2176194	best: 

In [7]:
exp_results = []

exp_results.append({
    "experiment": "exp27_cat_native_clean",
    "model": "CatBoost",
    "cv": "5-fold x 5-seed",
    "oof_rmse": 1.21469,
    "features": X.shape[1],
    "cat_features": len(cat_features),
    "notes": "Native categorical, categorical missing fixed"
})

pd.DataFrame(exp_results)

,experiment,model,cv,oof_rmse,features,cat_features,notes
0,exp27_cat_native_clean,CatBoost,5-fold x 5-seed,1.21469,23,7,"Native categorical, categorical missing fixed"


In [9]:
test_pred = np.load("test_exp27_cat_native_clean.npy")

submission = pd.DataFrame({
    "id": test["id"],
    "Degerlendirme Puani": test_pred
})

submission.to_csv("submission_exp27_cat_native_clean.csv", index=False)

submission.head()

,id,Degerlendirme Puani
0,1,6.064382
1,2,6.792259
2,3,3.055101
3,4,7.122827
4,5,3.753926


In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [11]:
oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")

In [12]:
import os

[f for f in os.listdir() if f.endswith(".npy")]

['oof_hgb_v3.npy',
 'test_hgb_v3.npy',
 'test_lgbm_v3.npy',
 'oof_exp27_cat_native_clean.npy',
 'test_exp27_cat_native_clean.npy',
 'oof_lgbm_v3.npy']

In [ ]:
# ============================================================
# DENEY 20E - TEK HÜCRE CLEAN RECREATE
# CatBoost v3 5-seed + HGB v3 + LightGBM v3 optimized blend
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool


# ------------------------------------------------------------
# Metric
# ------------------------------------------------------------

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert "X" in globals(), "X değişkeni yok. Önce feature matrix X oluşturulmalı."
assert "X_test" in globals(), "X_test değişkeni yok. Önce test feature matrix X_test oluşturulmalı."
assert "y" in globals(), "y değişkeni yok. Önce target y oluşturulmalı."

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("y shape:", y.shape)


# ------------------------------------------------------------
# Load HGB and LGBM OOF/Test predictions
# ------------------------------------------------------------

if "oof_hgb_v3" not in globals():
    assert os.path.exists("oof_hgb_v3.npy"), "oof_hgb_v3.npy bulunamadı."
    oof_hgb_v3 = np.load("oof_hgb_v3.npy")

if "test_hgb_v3" not in globals():
    assert os.path.exists("test_hgb_v3.npy"), "test_hgb_v3.npy bulunamadı."
    test_hgb_v3 = np.load("test_hgb_v3.npy")

if "oof_lgbm_v3" not in globals():
    assert os.path.exists("oof_lgbm_v3.npy"), "oof_lgbm_v3.npy bulunamadı."
    oof_lgbm_v3 = np.load("oof_lgbm_v3.npy")

if "test_lgbm_v3" not in globals():
    assert os.path.exists("test_lgbm_v3.npy"), "test_lgbm_v3.npy bulunamadı."
    test_lgbm_v3 = np.load("test_lgbm_v3.npy")


print("\nLoaded base models:")
print("HGB OOF:", oof_hgb_v3.shape, "| Test:", test_hgb_v3.shape, "| RMSE:", rmse(y, oof_hgb_v3))
print("LGBM OOF:", oof_lgbm_v3.shape, "| Test:", test_lgbm_v3.shape, "| RMSE:", rmse(y, oof_lgbm_v3))



cat_oof_file = "oof_cat_seed_ensemble_v3_5.npy"
cat_test_file = "test_cat_seed_ensemble_v3_5.npy"

if "oof_cat_seed_ensemble_v3_5" in globals() and "test_cat_seed_ensemble_v3_5" in globals():
    print("\nUsing existing CatBoost v3 5-seed variables from memory.")

elif os.path.exists(cat_oof_file) and os.path.exists(cat_test_file):
    print("\nLoading CatBoost v3 5-seed predictions from npy files.")
    oof_cat_seed_ensemble_v3_5 = np.load(cat_oof_file)
    test_cat_seed_ensemble_v3_5 = np.load(cat_test_file)

else:
    print("\nCatBoost v3 5-seed predictions not found. Training from scratch...")

    SEEDS = [42, 2024, 3407, 777, 999]
    N_SPLITS = 5


    X_cb = X.copy()
    X_test_cb = X_test.copy()

    cat_features = X_cb.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in cat_features:
        X_cb[col] = X_cb[col].fillna("missing").astype(str)
        X_test_cb[col] = X_test_cb[col].fillna("missing").astype(str)

    print("Categorical features:", cat_features)
    print("Number of categorical features:", len(cat_features))
    print("Remaining categorical NaN train:", X_cb[cat_features].isna().sum().sum())
    print("Remaining categorical NaN test:", X_test_cb[cat_features].isna().sum().sum())

    cat_oof_list_v3_5 = []
    cat_test_list_v3_5 = []
    cat_seed_scores_v3_5 = {}

    for seed in SEEDS:
        print("\n" + "=" * 70)
        print(f"CatBoost v3 5-seed training | Seed: {seed}")
        print("=" * 70)

        kf = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed
        )

        oof_pred = np.zeros(len(X_cb))
        test_pred = np.zeros(len(X_test_cb))
        fold_scores = []

        for fold, (train_idx, valid_idx) in enumerate(kf.split(X_cb, y), start=1):
            print(f"\nSeed {seed} | Fold {fold}")

            X_train = X_cb.iloc[train_idx]
            y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]

            X_valid = X_cb.iloc[valid_idx]
            y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

            train_pool = Pool(
                X_train,
                y_train,
                cat_features=cat_features
            )

            valid_pool = Pool(
                X_valid,
                y_valid,
                cat_features=cat_features
            )

            test_pool = Pool(
                X_test_cb,
                cat_features=cat_features
            )

            model = CatBoostRegressor(
                loss_function="RMSE",
                eval_metric="RMSE",
                iterations=5000,
                learning_rate=0.03,
                depth=6,
                l2_leaf_reg=3,
                random_seed=seed,
                verbose=500,
                early_stopping_rounds=300,
                allow_writing_files=False
            )

            model.fit(
                train_pool,
                eval_set=valid_pool,
                use_best_model=True
            )

            valid_pred = model.predict(valid_pool)
            fold_test_pred = model.predict(test_pool)

            oof_pred[valid_idx] = valid_pred
            test_pred += fold_test_pred / N_SPLITS

            fold_rmse = rmse(y_valid, valid_pred)
            fold_scores.append(fold_rmse)

            print(f"Seed {seed} | Fold {fold} RMSE: {fold_rmse:.6f}")

        seed_rmse = rmse(y, oof_pred)

        cat_oof_list_v3_5.append(oof_pred)
        cat_test_list_v3_5.append(test_pred)

        cat_seed_scores_v3_5[seed] = {
            "fold_scores": fold_scores,
            "seed_rmse": seed_rmse
        }

        print(f"\nSeed {seed} OOF RMSE: {seed_rmse:.6f}")

    assert len(cat_oof_list_v3_5) > 0, "CatBoost OOF listesi boş kaldı."
    assert len(cat_test_list_v3_5) > 0, "CatBoost test listesi boş kaldı."

    oof_cat_seed_ensemble_v3_5 = np.mean(cat_oof_list_v3_5, axis=0)
    test_cat_seed_ensemble_v3_5 = np.mean(cat_test_list_v3_5, axis=0)

    np.save(cat_oof_file, oof_cat_seed_ensemble_v3_5)
    np.save(cat_test_file, test_cat_seed_ensemble_v3_5)

    print("\nSaved:", cat_oof_file)
    print("Saved:", cat_test_file)


print("\nCatBoost v3 5-seed OOF:", rmse(y, oof_cat_seed_ensemble_v3_5))



assert oof_cat_seed_ensemble_v3_5.shape[0] == len(y), "CatBoost OOF shape hatalı."
assert oof_hgb_v3.shape[0] == len(y), "HGB OOF shape hatalı."
assert oof_lgbm_v3.shape[0] == len(y), "LGBM OOF shape hatalı."

assert test_cat_seed_ensemble_v3_5.shape[0] == len(X_test), "CatBoost test shape hatalı."
assert test_hgb_v3.shape[0] == len(X_test), "HGB test shape hatalı."
assert test_lgbm_v3.shape[0] == len(X_test), "LGBM test shape hatalı."




oof_matrix_v3_5 = np.column_stack([
    oof_cat_seed_ensemble_v3_5,
    oof_hgb_v3,
    oof_lgbm_v3
])

test_matrix_v3_5 = np.column_stack([
    test_cat_seed_ensemble_v3_5,
    test_hgb_v3,
    test_lgbm_v3
])

model_names_v3_5 = [
    "CatBoost v3 5 Seed Ensemble",
    "HGB v3",
    "LightGBM v3"
]


def blend_rmse_v3_5(weights):
    preds = oof_matrix_v3_5 @ weights
    return rmse(y, preds)


n_models = oof_matrix_v3_5.shape[1]
initial_weights = np.ones(n_models) / n_models

constraints = {
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
}

bounds = [(0, 1)] * n_models

result_v3_5 = minimize(
    blend_rmse_v3_5,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

best_weights_v3_5 = result_v3_5.x
best_oof_preds_v3_5 = oof_matrix_v3_5 @ best_weights_v3_5
best_test_preds_v3_5 = test_matrix_v3_5 @ best_weights_v3_5


print("\n" + "=" * 70)
print("DENEY 20E v3 5-seed optimized blend")
print("=" * 70)

print("Optimization success:", result_v3_5.success)
print("Best weights:")

for name, weight in zip(model_names_v3_5, best_weights_v3_5):
    print(f"{name}: {weight:.6f}")

print("\nBase model OOF scores:")
print("CatBoost v3 5 Seed Ensemble:", rmse(y, oof_cat_seed_ensemble_v3_5))
print("HGB v3:", rmse(y, oof_hgb_v3))
print("LightGBM v3:", rmse(y, oof_lgbm_v3))

print("\nDeney 20E v3 5-seed blend OOF:", rmse(y, best_oof_preds_v3_5))



np.save("oof_20e_recreated.npy", best_oof_preds_v3_5)
np.save("test_20e_recreated.npy", best_test_preds_v3_5)

print("\nSaved: oof_20e_recreated.npy")
print("Saved: test_20e_recreated.npy")



submission_20e_recreated = pd.DataFrame({
    "id": test["id"],
    "Degerlendirme Puani": best_test_preds_v3_5
})

submission_20e_recreated.to_csv("submission_20e_recreated.csv", index=False)

print("\nSubmission saved: submission_20e_recreated.csv")
print(submission_20e_recreated.head())
print("\nSubmission shape:", submission_20e_recreated.shape)
print("\nMissing values:")
print(submission_20e_recreated.isna().sum())

X shape: (56000, 23)
X_test shape: (24000, 23)
y shape: (56000,)

Loaded base models:
HGB OOF: (56000,) | Test: (24000,) | RMSE: 1.2293893343402866
LGBM OOF: (56000,) | Test: (24000,) | RMSE: 1.2310131834046825

CatBoost v3 5-seed predictions not found. Training from scratch...
Categorical features: ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
Number of categorical features: 7
Remaining categorical NaN train: 0
Remaining categorical NaN test: 0

CatBoost v3 5-seed training | Seed: 42

Seed 42 | Fold 1
0:	learn: 2.1962065	test: 2.2069443	best: 2.2069443 (0)	total: 8.11ms	remaining: 40.5s
500:	learn: 1.1958354	test: 1.2241326	best: 1.2241326 (500)	total: 3.67s	remaining: 33s
1000:	learn: 1.1711294	test: 1.2209564	best: 1.2209564 (1000)	total: 8.16s	remaining: 32.6s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 1.220786278
bestIteration = 1079

Shrink model to first 1080 iterations.
Seed 42 | Fold 1 RMSE: 1.220786

Seed 42 | F

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


oof_20e = np.load("oof_20e_recreated.npy")
test_20e = np.load("test_20e_recreated.npy")

oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")


print("20E recreated OOF:", rmse(y, oof_20e))
print("Exp27 OOF:", rmse(y, oof_27))


best_score = 999
best_w = None
blend_results = []

for w27 in np.arange(0, 1.001, 0.001):
    blend_oof = (1 - w27) * oof_20e + w27 * oof_27
    score = rmse(y, blend_oof)

    blend_results.append({
        "w_20e": 1 - w27,
        "w_27": w27,
        "rmse": score
    })

    if score < best_score:
        best_score = score
        best_w = w27

blend_results_df = pd.DataFrame(blend_results).sort_values("rmse")

print("Best weight for Exp27:", best_w)
print("Best weight for 20E:", 1 - best_w)
print("Best blend OOF:", best_score)

blend_results_df.head(20)

20E recreated OOF: 1.214421487375675
Exp27 OOF: 1.2146874339867157
Best weight for Exp27: 0.033
Best weight for 20E: 0.967
Best blend OOF: 1.2144211697595806


,w_20e,w_27,rmse
33,0.967,0.033,1.214421
34,0.966,0.034,1.214421
32,0.968,0.032,1.214421
35,0.965,0.035,1.214421
31,0.969,0.031,1.214421
36,0.964,0.036,1.214421
30,0.970,0.030,1.214421
37,0.963,0.037,1.214421
29,0.971,0.029,1.214421
38,0.962,0.038,1.214421


In [22]:
best_w_27 = 0.033
best_w_20e = 0.967

oof_20e = np.load("oof_20e_recreated.npy")
test_20e = np.load("test_20e_recreated.npy")

oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")

final_oof_blend = best_w_20e * oof_20e + best_w_27 * oof_27
final_test_blend = best_w_20e * test_20e + best_w_27 * test_27

print("Final blend OOF:", rmse(y, final_oof_blend))

Final blend OOF: 1.2144211697595806


In [25]:
submission_final_blend = pd.DataFrame({
    "id": test["id"],
    "Degerlendirme Puani": final_test_blend
})

submission_final_blend.to_csv("submission_20e_recreated_exp27_blend.csv", index=False)

submission_final_blend.head()

,id,Degerlendirme Puani
0,1,6.036028
1,2,6.776145
2,3,3.072168
3,4,7.136535
4,5,3.750549


In [26]:
submission_final_blend = pd.DataFrame({
    "id": test["id"],
    "bilissel_performans_skoru": final_test_blend
})

submission_final_blend.to_csv("submission_20e_recreated_exp27_blend_fixed.csv", index=False)

submission_final_blend.head()

,id,bilissel_performans_skoru
0,1,6.036028
1,2,6.776145
2,3,3.072168
3,4,7.136535
4,5,3.750549


In [27]:
print(submission_final_blend.shape)
print(submission_final_blend.isna().sum())
print(submission_final_blend.head())

(24000, 2)
id                           0
bilissel_performans_skoru    0
dtype: int64
   id  bilissel_performans_skoru
0   1                   6.036028
1   2                   6.776145
2   3                   3.072168
3   4                   7.136535
4   5                   3.750549


In [28]:
exp_results.append({
    "experiment": "20E_recreated_exp27_blend",
    "model": "20E recreated + Exp27",
    "local_oof_rmse": 1.2144211697595806,
    "public_lb": 1.20268,
    "weights": "20E=0.967, Exp27=0.033",
    "submission": "submission_20e_recreated_exp27_blend_fixed.csv",
    "notes": "Improved public score from 1.20273 to 1.20268"
})

pd.DataFrame(exp_results)

,experiment,model,cv,oof_rmse,features,cat_features,notes,local_oof_rmse,public_lb,weights,submission
0,exp27_cat_native_clean,CatBoost,5-fold x 5-seed,1.21469,23.0,7.0,"Native categorical, categorical missing fixed",NaN,NaN,NaN,NaN
1,20E_recreated_exp27_blend,20E recreated + Exp27,NaN,NaN,NaN,NaN,Improved public score from 1.20273 to 1.20268,1.214421,1.20268,"20E=0.967, Exp27=0.033",submission_20e_recreated_exp27_blend_fixed.csv


In [ ]:
# ============================================================
# DENEY 28
# CatBoost 5-seed + Feature Engineering + Best Blend Comparison
# ============================================================

import os
import numpy as np
import pandas as pd

from itertools import combinations
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool




def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))



assert "X" in globals(), "X yok."
assert "X_test" in globals(), "X_test yok."
assert "y" in globals(), "y yok."
assert "test" in globals(), "test yok."

print("Original X:", X.shape)
print("Original X_test:", X_test.shape)
print("y:", y.shape)



def safe_div(a, b):
    return a / (b.replace(0, np.nan) + 1e-6)


def find_col(columns, keywords):
    """
    Kolon isimlerinde verilen keyword'leri arar.
    Örn: keywords=["uyku", "süre"] -> uyku_suresi gibi kolonları yakalamaya çalışır.
    """
    cols_lower = {c: c.lower() for c in columns}

    for original_col, lower_col in cols_lower.items():
        ok = True
        for kw in keywords:
            if kw.lower() not in lower_col:
                ok = False
                break
        if ok:
            return original_col

    return None


def add_if_exists_ratio(X_train_fe, X_test_fe, numerator_col, denominator_col, new_col):
    if numerator_col is not None and denominator_col is not None:
        X_train_fe[new_col] = safe_div(X_train_fe[numerator_col], X_train_fe[denominator_col])
        X_test_fe[new_col] = safe_div(X_test_fe[numerator_col], X_test_fe[denominator_col])
        print("Added ratio:", new_col)


def add_if_exists_interaction(X_train_fe, X_test_fe, col1, col2, new_col):
    if col1 is not None and col2 is not None:
        X_train_fe[new_col] = X_train_fe[col1] * X_train_fe[col2]
        X_test_fe[new_col] = X_test_fe[col1] * X_test_fe[col2]
        print("Added interaction:", new_col)



def make_exp28_features(X_base, X_test_base):
    X_fe = X_base.copy()
    X_test_fe = X_test_base.copy()

    for df in [X_fe, X_test_fe]:
        if "id" in df.columns:
            df.drop(columns=["id"], inplace=True)

    cat_cols = X_fe.select_dtypes(include=["object", "category"]).columns.tolist()

    print("\nOriginal categorical columns:", cat_cols)

    for col in cat_cols:
        X_fe[col] = X_fe[col].fillna("missing").astype(str)
        X_test_fe[col] = X_test_fe[col].fillna("missing").astype(str)


    preferred_cat_pairs = [
        ("kronotip", "gun_tipi"),
        ("mevsim", "kronotip"),
        ("meslek", "gun_tipi"),
        ("ruh_sagligi_durumu", "gun_tipi"),
        ("cinsiyet", "meslek"),
        ("ulke", "meslek"),
        ("mevsim", "gun_tipi"),
        ("ruh_sagligi_durumu", "kronotip"),
    ]

    for c1, c2 in preferred_cat_pairs:
        if c1 in X_fe.columns and c2 in X_fe.columns:
            new_col = f"{c1}__{c2}"
            X_fe[new_col] = X_fe[c1].astype(str) + "__" + X_fe[c2].astype(str)
            X_test_fe[new_col] = X_test_fe[c1].astype(str) + "__" + X_test_fe[c2].astype(str)
            print("Added cat combo:", new_col)


    num_cols = X_fe.select_dtypes(include=[np.number]).columns.tolist()

    print("\nOriginal numeric columns:", num_cols)

    X_fe["num_missing_count"] = X_fe[num_cols].isna().sum(axis=1)
    X_test_fe["num_missing_count"] = X_test_fe[num_cols].isna().sum(axis=1)

    X_fe["num_mean"] = X_fe[num_cols].mean(axis=1)
    X_test_fe["num_mean"] = X_test_fe[num_cols].mean(axis=1)

    X_fe["num_std"] = X_fe[num_cols].std(axis=1)
    X_test_fe["num_std"] = X_test_fe[num_cols].std(axis=1)

    X_fe["num_min"] = X_fe[num_cols].min(axis=1)
    X_test_fe["num_min"] = X_test_fe[num_cols].min(axis=1)

    X_fe["num_max"] = X_fe[num_cols].max(axis=1)
    X_test_fe["num_max"] = X_test_fe[num_cols].max(axis=1)

    X_fe["num_range"] = X_fe["num_max"] - X_fe["num_min"]
    X_test_fe["num_range"] = X_test_fe["num_max"] - X_test_fe["num_min"]


    columns = X_fe.columns.tolist()

    sleep_col = find_col(columns, ["uyku"])
    screen_col = find_col(columns, ["ekran"])
    physical_col = find_col(columns, ["fizik"])
    stress_col = find_col(columns, ["stres"])
    age_col = find_col(columns, ["yaş"]) or find_col(columns, ["yas"])
    social_col = find_col(columns, ["sosyal"])
    work_col = find_col(columns, ["çalış"]) or find_col(columns, ["calis"])
    caffeine_col = find_col(columns, ["kafe"]) or find_col(columns, ["kahve"])

    print("\nDetected special columns:")
    print("sleep_col:", sleep_col)
    print("screen_col:", screen_col)
    print("physical_col:", physical_col)
    print("stress_col:", stress_col)
    print("age_col:", age_col)
    print("social_col:", social_col)
    print("work_col:", work_col)
    print("caffeine_col:", caffeine_col)

    add_if_exists_ratio(X_fe, X_test_fe, screen_col, sleep_col, "screen_sleep_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, physical_col, sleep_col, "physical_sleep_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, social_col, screen_col, "social_screen_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, work_col, sleep_col, "work_sleep_ratio")
    add_if_exists_ratio(X_fe, X_test_fe, caffeine_col, sleep_col, "caffeine_sleep_ratio")

    add_if_exists_interaction(X_fe, X_test_fe, stress_col, screen_col, "stress_x_screen")
    add_if_exists_interaction(X_fe, X_test_fe, stress_col, sleep_col, "stress_x_sleep")
    add_if_exists_interaction(X_fe, X_test_fe, stress_col, physical_col, "stress_x_physical")
    add_if_exists_interaction(X_fe, X_test_fe, screen_col, physical_col, "screen_x_physical")
    add_if_exists_interaction(X_fe, X_test_fe, age_col, sleep_col, "age_x_sleep")
    add_if_exists_interaction(X_fe, X_test_fe, age_col, screen_col, "age_x_screen")

    # --------------------------------------------------------
    # 4) Yaş bin feature'ı varsa
    # --------------------------------------------------------

    if age_col is not None:
        X_fe["age_bin"] = pd.cut(
            X_fe[age_col],
            bins=[-np.inf, 18, 25, 35, 45, 60, np.inf],
            labels=["age_0_18", "age_19_25", "age_26_35", "age_36_45", "age_46_60", "age_60_plus"]
        ).astype(str)

        X_test_fe["age_bin"] = pd.cut(
            X_test_fe[age_col],
            bins=[-np.inf, 18, 25, 35, 45, 60, np.inf],
            labels=["age_0_18", "age_19_25", "age_26_35", "age_36_45", "age_46_60", "age_60_plus"]
        ).astype(str)

        print("Added categorical age_bin")

    # --------------------------------------------------------
    # 5) Inf temizliği
    # --------------------------------------------------------

    X_fe = X_fe.replace([np.inf, -np.inf], np.nan)
    X_test_fe = X_test_fe.replace([np.inf, -np.inf], np.nan)

    # Son kategorik kolonlar
    final_cat_cols = X_fe.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in final_cat_cols:
        X_fe[col] = X_fe[col].fillna("missing").astype(str)
        X_test_fe[col] = X_test_fe[col].fillna("missing").astype(str)

    return X_fe, X_test_fe, final_cat_cols


# ------------------------------------------------------------
# Create Exp28 features
# ------------------------------------------------------------

X28, X_test28, cat_features28 = make_exp28_features(X, X_test)

print("\nExp28 X shape:", X28.shape)
print("Exp28 X_test shape:", X_test28.shape)
print("Exp28 categorical feature count:", len(cat_features28))
print("Exp28 categorical features:", cat_features28)
print("Remaining categorical NaN train:", X28[cat_features28].isna().sum().sum())
print("Remaining categorical NaN test:", X_test28[cat_features28].isna().sum().sum())


# ------------------------------------------------------------
# Train CatBoost 5-seed on Exp28 features
# ------------------------------------------------------------

EXP_NAME = "exp28_cat_fe_clean"
SEEDS = [42, 2024, 3407, 777, 999]
N_SPLITS = 5

cat_oof_list_exp28 = []
cat_test_list_exp28 = []
cat_seed_scores_exp28 = {}

for seed in SEEDS:
    print("\n" + "=" * 70)
    print(f"{EXP_NAME} | Seed: {seed}")
    print("=" * 70)

    kf = KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    oof_pred = np.zeros(len(X28))
    test_pred = np.zeros(len(X_test28))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X28, y), start=1):
        print(f"\nSeed {seed} | Fold {fold}")

        X_train = X28.iloc[train_idx]
        y_train = y.iloc[train_idx] if hasattr(y, "iloc") else y[train_idx]

        X_valid = X28.iloc[valid_idx]
        y_valid = y.iloc[valid_idx] if hasattr(y, "iloc") else y[valid_idx]

        train_pool = Pool(
            X_train,
            y_train,
            cat_features=cat_features28
        )

        valid_pool = Pool(
            X_valid,
            y_valid,
            cat_features=cat_features28
        )

        test_pool = Pool(
            X_test28,
            cat_features=cat_features28
        )

        model = CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            iterations=6000,
            learning_rate=0.025,
            depth=6,
            l2_leaf_reg=4,
            random_seed=seed,
            verbose=500,
            early_stopping_rounds=350,
            allow_writing_files=False
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True
        )

        valid_pred = model.predict(valid_pool)
        fold_test_pred = model.predict(test_pool)

        oof_pred[valid_idx] = valid_pred
        test_pred += fold_test_pred / N_SPLITS

        fold_rmse = rmse(y_valid, valid_pred)
        fold_scores.append(fold_rmse)

        print(f"Seed {seed} | Fold {fold} RMSE: {fold_rmse:.6f}")

    seed_rmse = rmse(y, oof_pred)

    cat_oof_list_exp28.append(oof_pred)
    cat_test_list_exp28.append(test_pred)

    cat_seed_scores_exp28[seed] = {
        "fold_scores": fold_scores,
        "seed_rmse": seed_rmse
    }

    print(f"\nSeed {seed} OOF RMSE: {seed_rmse:.6f}")


assert len(cat_oof_list_exp28) > 0, "Exp28 OOF listesi boş kaldı."
assert len(cat_test_list_exp28) > 0, "Exp28 test listesi boş kaldı."

oof_exp28_cat_fe_clean = np.mean(cat_oof_list_exp28, axis=0)
test_exp28_cat_fe_clean = np.mean(cat_test_list_exp28, axis=0)

exp28_score = rmse(y, oof_exp28_cat_fe_clean)

print("\n" + "=" * 70)
print(f"{EXP_NAME} OOF RMSE:", exp28_score)
print("=" * 70)

np.save("oof_exp28_cat_fe_clean.npy", oof_exp28_cat_fe_clean)
np.save("test_exp28_cat_fe_clean.npy", test_exp28_cat_fe_clean)

print("Saved: oof_exp28_cat_fe_clean.npy")
print("Saved: test_exp28_cat_fe_clean.npy")


# ------------------------------------------------------------
# Compare with current best
# Current best = 20E recreated + Exp27 blend
# ------------------------------------------------------------

oof_20e = np.load("oof_20e_recreated.npy")
test_20e = np.load("test_20e_recreated.npy")

oof_27 = np.load("oof_exp27_cat_native_clean.npy")
test_27 = np.load("test_exp27_cat_native_clean.npy")

current_best_oof = 0.967 * oof_20e + 0.033 * oof_27
current_best_test = 0.967 * test_20e + 0.033 * test_27

current_best_score = rmse(y, current_best_oof)

print("\nCurrent best OOF:", current_best_score)
print("Exp28 OOF:", exp28_score)


# ------------------------------------------------------------
# Blend current best with Exp28
# ------------------------------------------------------------

best_score = 999
best_w_exp28 = None
blend_results_exp28 = []

for w_exp28 in np.arange(0, 1.001, 0.001):
    blend_oof = (1 - w_exp28) * current_best_oof + w_exp28 * oof_exp28_cat_fe_clean
    score = rmse(y, blend_oof)

    blend_results_exp28.append({
        "w_current_best": 1 - w_exp28,
        "w_exp28": w_exp28,
        "rmse": score
    })

    if score < best_score:
        best_score = score
        best_w_exp28 = w_exp28

blend_results_exp28_df = pd.DataFrame(blend_results_exp28).sort_values("rmse")

print("\n" + "=" * 70)
print("Current Best + Exp28 Blend")
print("=" * 70)

print("Best weight for Exp28:", best_w_exp28)
print("Best weight for current best:", 1 - best_w_exp28)
print("Best blend OOF:", best_score)

display(blend_results_exp28_df.head(20))


# ------------------------------------------------------------
# Save final candidate submission
# ------------------------------------------------------------

final_oof_exp28_blend = (1 - best_w_exp28) * current_best_oof + best_w_exp28 * oof_exp28_cat_fe_clean
final_test_exp28_blend = (1 - best_w_exp28) * current_best_test + best_w_exp28 * test_exp28_cat_fe_clean

print("\nFinal Exp28 blend OOF:", rmse(y, final_oof_exp28_blend))

submission_exp28_blend = pd.DataFrame({
    "id": test["id"],
    "bilissel_performans_skoru": final_test_exp28_blend
})

submission_exp28_blend.to_csv("submission_exp28_current_best_blend.csv", index=False)

print("\nSaved: submission_exp28_current_best_blend.csv")
print(submission_exp28_blend.head())
print("\nShape:", submission_exp28_blend.shape)
print("\nMissing:")
print(submission_exp28_blend.isna().sum())
print("\nPrediction describe:")
print(submission_exp28_blend["bilissel_performans_skoru"].describe())

Original X: (56000, 23)
Original X_test: (24000, 23)
y: (56000,)

Original categorical columns: ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
Added cat combo: kronotip__gun_tipi
Added cat combo: mevsim__kronotip
Added cat combo: meslek__gun_tipi
Added cat combo: ruh_sagligi_durumu__gun_tipi
Added cat combo: cinsiyet__meslek
Added cat combo: ulke__meslek
Added cat combo: mevsim__gun_tipi
Added cat combo: ruh_sagligi_durumu__kronotip

Original numeric columns: ['yas', 'vucut_kitle_indeksi', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg', 'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk', 'stres_skoru', 'gunluk_calisma_saati', 'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat']

Detected special columns:
sleep_col: derin_uyku_yuzdesi
screen_col: uyku_oncesi_ekran_suresi_dk
physical_col: None
stress_col: stres_skoru
age_col: yas
social_

,w_current_best,w_exp28,rmse
658,0.342,0.658,1.213269
657,0.343,0.657,1.213269
659,0.341,0.659,1.213269
656,0.344,0.656,1.213269
660,0.340,0.660,1.213269
655,0.345,0.655,1.213269
661,0.339,0.661,1.213269
654,0.346,0.654,1.213269
662,0.338,0.662,1.213269
653,0.347,0.653,1.213269



Final Exp28 blend OOF: 1.2132691035554823

Saved: submission_exp28_current_best_blend.csv
   id  bilissel_performans_skoru
0   1                   6.001816
1   2                   6.737925
2   3                   3.044147
3   4                   7.158813
4   5                   3.694694

Shape: (24000, 2)

Missing:
id                           0
bilissel_performans_skoru    0
dtype: int64

Prediction describe:
count    24000.000000
mean         5.937791
std          1.864037
min         -0.229105
25%          4.645149
50%          6.045080
75%          7.321425
max         10.761977
Name: bilissel_performans_skoru, dtype: float64


In [30]:
submission_exp28_blend_clipped = submission_exp28_blend.copy()

submission_exp28_blend_clipped["bilissel_performans_skoru"] = (
    submission_exp28_blend_clipped["bilissel_performans_skoru"]
    .clip(0, 10)
)

submission_exp28_blend_clipped.to_csv(
    "submission_exp28_current_best_blend_clipped.csv",
    index=False
)

print(submission_exp28_blend_clipped.shape)
print(submission_exp28_blend_clipped.isna().sum())
print(submission_exp28_blend_clipped["bilissel_performans_skoru"].describe())

(24000, 2)
id                           0
bilissel_performans_skoru    0
dtype: int64
count    24000.000000
mean         5.937440
std          1.863110
min          0.000000
25%          4.645149
50%          6.045080
75%          7.321425
max         10.000000
Name: bilissel_performans_skoru, dtype: float64
